# C00_gold_check__CPU — Pilot C

Applies every task's gold patch AND no patch in the scorer's verification path. A task is valid only if gold passes and no-patch fails. Records why gold fails (missing module / fixture / test failure), counts fastapi tasks failing on `inline_snapshot`/`dirty_equals`, and writes the pre-registered task order and smoke set used by C01 and C03.

- **Accelerator:** None (CPU)
- **Internet:** Off
- **Inputs to attach:** competition `gemma-4-developer-agent`; dataset `metric/gemma-4-developer-agent-wheelhouse`
- **Earlier output needed:** none (this notebook runs first).
- **Expected run time:** ≈ 60–90 min CPU (129 tasks, each verified with the gold patch and with no patch); 0 GPU h
- **Bring back:** `/kaggle/working/C00_gold_check.json` → commit to `results/` on branch `pilot/c-bottleneck`. Leave `/kaggle/working/private/` on Kaggle (competition-derived).

**How to run it in the right environment.** These notebooks must run in the **official starter's pinned Kaggle environment (Python 3.12)**. New Kaggle notebooks default to Python 3.13, and pip then rejects the wheelhouse's cp312 wheels.

1. Open the official starter: https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent and click **Copy & Edit**. The copy keeps the starter's pinned environment (Python 3.12) and its inputs: the competition, the wheelhouse and the four Gemma 4 models.
2. In the copy: **File → Import Notebook** → upload `notebooks/C00_gold_check__CPU.ipynb` from branch `pilot/c-bottleneck`. This replaces the cells; the environment and inputs stay.
3. Rename the notebook (title, top left) to `C00_gold_check__CPU`.
4. Right panel → **Settings**: Environment = **Pin to original environment** (do not switch to "Always use latest environment"); Accelerator = **None**; Internet = **Off**.
5. Inputs: No earlier notebook output is needed. The Gemma 4 models that came with the copy are not needed (you can remove them).
6. **Save Version → Save & Run All**. The first code cell prints the Python version and stops within seconds if it is not 3.12. In that case the environment is not the starter's: redo steps 1–4.

In [ ]:
# Environment check (must stay the first code cell)
import sys
print('Python', sys.version)
EXPECTED = (3, 12)   # the official starter's pinned Kaggle environment; the wheelhouse is built for cp312
if sys.version_info[:2] != EXPECTED:
    raise RuntimeError(
        f'WRONG ENVIRONMENT: this session runs Python {sys.version_info[0]}.{sys.version_info[1]}, these notebooks need '
        f'Python {EXPECTED[0]}.{EXPECTED[1]}. Open the official starter (ryanholbrook/getting-started-gemma-4-developer-agent), '
        'click Copy & Edit, then File -> Import Notebook with this .ipynb, and keep Settings -> Environment = '
        '"Pin to original environment". Stopped before installing anything.')
print('environment OK')

In [ ]:
import os
os.makedirs('/kaggle/working/pilot/common', exist_ok=True)
os.makedirs('/kaggle/working/pilot/pilot_c', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/pilot/common/__init__.py
# (empty module)

In [ ]:
%%writefile /kaggle/working/pilot/common/diffs.py
"""Unified-diff parsing helpers (no task data, no pilot-specific logic).

Parses the `--- a/X / +++ b/X / @@ ... @@` diffs used in the competition's
`patch` and `test_patch` fields into per-file change records with old-side
line numbers, so callers can map changes onto the base snapshot.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field

_HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")

# Mirrors the scorer's notion of protected test files (HARNESS_README §8.1).
_TEST_DIR_PARTS = {"tests", "test", "testing"}


def is_test_path(path: str) -> bool:
    parts = path.split("/")
    name = parts[-1]
    if name.startswith("test_") and name.endswith(".py"):
        return True
    if name.endswith("_test.py"):
        return True
    return name.endswith(".py") and any(p in _TEST_DIR_PARTS for p in parts[:-1])


@dataclass
class Insertion:
    """Lines added after old line `after_old` (0 = at file start)."""

    after_old: int
    lines: list[str]


@dataclass
class FileDiff:
    old_path: str | None
    new_path: str | None
    removed_old_lines: list[int] = field(default_factory=list)  # old-side line numbers deleted/replaced
    insertions: list[Insertion] = field(default_factory=list)
    hunks: int = 0
    added: int = 0
    removed: int = 0
    all_hunks_from_zero: bool = True  # every hunk is "@@ -0,0 ..." (file absent on old side)

    @property
    def path(self) -> str:
        return self.new_path or self.old_path or ""

    @property
    def maybe_new_file(self) -> bool:
        return self.old_path is None or (self.hunks > 0 and self.all_hunks_from_zero)

    @property
    def deleted_file(self) -> bool:
        return self.new_path is None


def _strip_prefix(p: str) -> str | None:
    p = p.split("\t", 1)[0].strip()
    if p == "/dev/null":
        return None
    if p.startswith('"') and p.endswith('"'):
        p = p[1:-1]
    if p.startswith(("a/", "b/")):
        p = p[2:]
    return p


def parse_unified_diff(text: str) -> list[FileDiff]:
    """Parse a unified diff into FileDiff records (robust to missing `diff --git` headers)."""
    files: list[FileDiff] = []
    cur: FileDiff | None = None
    lines = text.splitlines()
    i = 0
    old_ln = new_ln = 0
    rem_old = rem_new = 0
    pending: Insertion | None = None

    def flush_pending() -> None:
        nonlocal pending
        if pending is not None and cur is not None:
            cur.insertions.append(pending)
        pending = None

    while i < len(lines):
        line = lines[i]
        in_hunk = rem_old > 0 or rem_new > 0
        if not in_hunk and line.startswith("--- ") and i + 1 < len(lines) and lines[i + 1].startswith("+++ "):
            flush_pending()
            cur = FileDiff(old_path=_strip_prefix(line[4:]), new_path=_strip_prefix(lines[i + 1][4:]))
            files.append(cur)
            i += 2
            continue
        if not in_hunk and line.startswith("@@ ") and cur is not None:
            flush_pending()
            m = _HUNK_RE.match(line)
            if m:
                o_start = int(m.group(1))
                o_cnt = int(m.group(2)) if m.group(2) is not None else 1
                n_cnt = int(m.group(4)) if m.group(4) is not None else 1
                cur.hunks += 1
                if not (o_start == 0 and o_cnt == 0):
                    cur.all_hunks_from_zero = False
                # old_ln = number of the last old line consumed so far
                old_ln = o_start - 1 if o_cnt > 0 else o_start
                rem_old, rem_new = o_cnt, n_cnt
            i += 1
            continue
        if in_hunk and cur is not None:
            if line.startswith("\\"):  # "\ No newline at end of file"
                i += 1
                continue
            tag = line[:1]
            body = line[1:]
            if tag == "+":
                if pending is None or pending.after_old != old_ln:
                    flush_pending()
                    pending = Insertion(after_old=old_ln, lines=[])
                pending.lines.append(body)
                cur.added += 1
                rem_new -= 1
            elif tag == "-":
                flush_pending()
                old_ln += 1
                cur.removed_old_lines.append(old_ln)
                cur.removed += 1
                rem_old -= 1
            else:  # context (' ' or empty line inside hunk)
                flush_pending()
                old_ln += 1
                rem_old -= 1
                rem_new -= 1
            i += 1
            continue
        i += 1
    flush_pending()
    return files


_ADDED_TEST_DEF = re.compile(r"^\+\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)
_REMOVED_TEST_DEF = re.compile(r"^-\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)
_HUNK_HEADER_TEST = re.compile(r"^@@[^@]+@@\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\b", re.M)
_CONTEXT_TEST_DEF = re.compile(r"^ \s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)


def required_test_names(test_patch: str) -> list[str]:
    """Test function names the scorer requires to pass, derived from test_patch.

    Re-implements the documented rule of the scorer (swegemma 0.2.11,
    `_extract_test_functions_from_patch`): added `def test_*` lines if any;
    otherwise test functions named in hunk headers / context, minus removed ones.
    Verified equal to the scorer's function on all 129 dev tasks (see
    pilot_c/check_test_names.py).
    """
    if not test_patch:
        return []
    added = _ADDED_TEST_DEF.findall(test_patch)
    if added:
        return list(dict.fromkeys(added))
    removed = set(_REMOVED_TEST_DEF.findall(test_patch))
    headers = _HUNK_HEADER_TEST.findall(test_patch)
    context = _CONTEXT_TEST_DEF.findall(test_patch)
    ctx = set(context)
    out = []
    for fn in dict.fromkeys(headers + context):
        if fn in removed:
            continue
        if fn not in ctx and (any(r.startswith(fn) for r in removed) or any(c != fn and c.startswith(fn) for c in ctx)):
            continue
        out.append(fn)
    return out

In [ ]:
%%writefile /kaggle/working/pilot/common/locate.py
"""Map diff changes onto enclosing Python scopes (def/class) of the base file.

Given the base-commit source of a file and a parsed FileDiff, returns the
innermost enclosing function/class for every changed region, using `ast` on
the base file. Deleted/replaced lines map by line number; inserted blocks map
by their anchor line plus indentation (an insertion indented less than a
scope's body belongs to the parent scope).
"""
from __future__ import annotations

import ast
from dataclasses import dataclass

from .diffs import FileDiff


@dataclass(frozen=True)
class Scope:
    qualname: str  # e.g. "APIRoute.__init__" (relative to module)
    kind: str  # "function" | "class"
    start: int  # first line incl. decorators
    def_line: int
    end: int
    body_indent: int
    depth: int
    header_indent: int = 0


def module_path(file_path: str) -> str:
    """'src/requests/models.py' -> 'requests.models'; 'pkg/__init__.py' -> 'pkg'."""
    p = file_path[:-3] if file_path.endswith(".py") else file_path
    parts = [x for x in p.split("/") if x]
    if parts and parts[0] in ("src", "lib"):
        parts = parts[1:]
    if parts and parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts)


def collect_scopes(source: str) -> list[Scope]:
    tree = ast.parse(source)
    scopes: list[Scope] = []

    def visit(node: ast.AST, prefix: str, depth: int) -> None:
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f"{prefix}.{child.name}" if prefix else child.name
                start = min([child.lineno] + [d.lineno for d in child.decorator_list])
                body_indent = child.body[0].col_offset if child.body else child.col_offset + 4
                kind = "class" if isinstance(child, ast.ClassDef) else "function"
                scopes.append(Scope(q, kind, start, child.lineno, child.end_lineno or child.lineno, body_indent, depth, child.col_offset))
                visit(child, q, depth + 1)
            else:
                visit(child, prefix, depth)

    visit(tree, "", 0)
    return scopes


def _innermost(scopes: list[Scope], line: int) -> list[Scope]:
    """Scopes containing `line`, outermost first."""
    return sorted((s for s in scopes if s.start <= line <= s.end), key=lambda s: s.depth)


def _indent(s: str) -> int:
    return len(s) - len(s.lstrip(" \t"))


def report_scope(chain: list[Scope]) -> Scope | None:
    """Collapse a containment chain to what a developer would name:
    the outermost function (methods count as functions), or the innermost class
    if the change is in a class body outside any method."""
    if not chain:
        return None
    for s in chain:
        if s.kind == "function":
            return s
    return chain[-1]


def changed_scopes(source: str, fd: FileDiff) -> tuple[list[Scope], bool]:
    """Return (scopes touched by the diff, touches_module_level_code)."""
    scopes = collect_scopes(source)
    src_lines = source.splitlines()
    hits: dict[str, Scope] = {}
    module_level = False

    def last_code_line_at_or_before(n: int) -> int:
        while n >= 1 and (n > len(src_lines) or not src_lines[n - 1].strip()):
            n -= 1
        return n

    for ln in fd.removed_old_lines:
        s = report_scope(_innermost(scopes, ln))
        if s is None:
            module_level = True
        else:
            hits.setdefault(s.qualname, s)

    removed = set(fd.removed_old_lines)
    for ins in fd.insertions:
        code = [x for x in ins.lines if x.strip()]
        if not code:
            continue
        ind = _indent(code[0])
        if ins.after_old in removed:
            # replacement: the new lines take the place of removed ones, so they may
            # sit at a scope's header level (e.g. an edited decorator or signature)
            chain = [s for s in _innermost(scopes, ins.after_old) if s.header_indent <= ind]
        else:
            anchor = last_code_line_at_or_before(ins.after_old)
            chain = [s for s in _innermost(scopes, anchor) if s.body_indent <= ind] if anchor >= 1 else []
        s = report_scope(chain)
        if s is None:
            module_level = True
        else:
            hits.setdefault(s.qualname, s)

    ordered = sorted(hits.values(), key=lambda s: s.def_line)
    return ordered, module_level

In [ ]:
%%writefile /kaggle/working/pilot/common/tasks.py
"""Task loading and gold-location extraction (reusable; no pilot-specific logic).

Nothing here writes task data anywhere: callers decide where outputs go.
Competition data (tasks.jsonl, snapshots) must stay out of git.
"""
from __future__ import annotations

import json
import tarfile
from dataclasses import dataclass, field, asdict
from pathlib import Path

from .diffs import FileDiff, is_test_path, parse_unified_diff, required_test_names
from .locate import changed_scopes, module_path


def load_tasks(path: str | Path) -> list[dict]:
    out, seen = [], set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                t = json.loads(line)
                if t["instance_id"] not in seen:
                    seen.add(t["instance_id"])
                    out.append(t)
    return out


def read_snapshot_members(tgz: str | Path, wanted: set[str]) -> dict[str, str]:
    """Return {path: text} for the wanted repo-relative paths, reading the archive in one pass.
    Never extracts to disk and never executes anything from the archive."""
    found: dict[str, str] = {}
    want = set(wanted)
    with tarfile.open(tgz, mode="r|gz") as tf:
        for m in tf:
            name = m.name[2:] if m.name.startswith("./") else m.name
            if name in want and m.isfile():
                fh = tf.extractfile(m)
                if fh is not None:
                    found[name] = fh.read().decode("utf-8", errors="replace")
                if len(found) == len(want):
                    break
    return found


@dataclass
class GoldFunction:
    file: str
    qualname: str  # relative to module, e.g. "APIRoute.__init__"
    dotted: str  # e.g. "fastapi.routing.APIRoute.__init__" (graph-node style)
    kind: str  # "function" | "class"
    def_line: int


@dataclass
class GoldLocations:
    instance_id: str
    repo: str
    src_files: list[str] = field(default_factory=list)  # non-test .py files changed by the gold patch
    new_src_files: list[str] = field(default_factory=list)  # subset absent from the base snapshot
    other_files: list[str] = field(default_factory=list)  # non-.py or test files in the gold patch
    functions: list[GoldFunction] = field(default_factory=list)
    module_level_files: list[str] = field(default_factory=list)  # files with changes outside any def/class
    test_files: list[str] = field(default_factory=list)
    new_test_files: list[str] = field(default_factory=list)  # test files absent from the base snapshot
    test_names: list[str] = field(default_factory=list)  # scorer-required test functions
    gold_added: int = 0
    gold_removed: int = 0
    parse_errors: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return asdict(self)


def diff_scopes(diff_text: str, snapshot_tgz: str | Path, texts: dict[str, str] | None = None) -> list[str]:
    """Dotted names of the def/class scopes (in the base snapshot) changed by any diff,
    e.g. an agent's patch. Module-level changes are reported as '<module>:<path>'."""
    fds = [fd for fd in parse_unified_diff(diff_text) if fd.path.endswith(".py") and fd.old_path]
    if texts is None:
        texts = read_snapshot_members(snapshot_tgz, {fd.path for fd in fds})
    out: list[str] = []
    for fd in fds:
        if fd.path not in texts:
            continue
        try:
            scopes, mod_level = changed_scopes(texts[fd.path], fd)
        except SyntaxError:
            continue
        mod = module_path(fd.path)
        out += [f"{mod}.{s.qualname}" if mod else s.qualname for s in scopes]
        if mod_level:
            out.append(f"<module>:{fd.path}")
    return out


def gold_locations(task: dict, snapshot_tgz: str | Path | None) -> GoldLocations:
    """Derive gold file(s), enclosing function(s) and required test names for one task.

    Uses only *locations* from the gold patch (paths, line numbers) and test
    names from test_patch; the base-file AST comes from the snapshot.
    If snapshot_tgz is None, function-level fields stay empty.
    """
    g = GoldLocations(task["instance_id"], task["repo"])
    fdiffs: list[FileDiff] = parse_unified_diff(task["patch"])
    tdiffs: list[FileDiff] = parse_unified_diff(task["test_patch"])
    py_src = [fd for fd in fdiffs if fd.path.endswith(".py") and not is_test_path(fd.path)]
    g.src_files = list(dict.fromkeys(fd.path for fd in py_src))
    g.other_files = [fd.path for fd in fdiffs if fd not in py_src]
    g.gold_added = sum(fd.added for fd in fdiffs)
    g.gold_removed = sum(fd.removed for fd in fdiffs)
    g.test_files = list(dict.fromkeys(fd.path for fd in tdiffs if fd.new_path))
    g.test_names = required_test_names(task["test_patch"])

    if snapshot_tgz is None:
        return g
    wanted = {fd.path for fd in py_src if fd.old_path} | set(g.test_files)
    texts = read_snapshot_members(snapshot_tgz, wanted)
    g.new_test_files = [p for p in g.test_files if p not in texts]
    for fd in py_src:
        if fd.path not in texts:
            g.new_src_files.append(fd.path)
            continue
        try:
            scopes, mod_level = changed_scopes(texts[fd.path], fd)
        except SyntaxError as e:  # pragma: no cover - base files should parse
            g.parse_errors.append(f"{fd.path}: {e}")
            continue
        mod = module_path(fd.path)
        for s in scopes:
            g.functions.append(GoldFunction(fd.path, s.qualname, f"{mod}.{s.qualname}" if mod else s.qualname, s.kind, s.def_line))
        if mod_level:
            g.module_level_files.append(fd.path)
    return g

In [ ]:
%%writefile /kaggle/working/pilot/common/trace.py
"""Parse swegemma/adk-eval-core session traces into a flat event list.

Supports the ATIF v1.7 JSON written by `SessionTrace.save()` (the harness
default, `results_dir/traces/trace_<id>.json`) and the legacy
`{"entries": [...]}` format. Mirrors `SessionTrace.from_dict` in
adk-eval-core 0.1.0 without importing it, so this works on any machine.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any


@dataclass
class Event:
    type: str  # tool_call | tool_response | thinking | text | final | continuation_nudge | compaction | ...
    author: str = ""
    elapsed: float = 0.0
    tool: str = ""
    args: dict[str, Any] = field(default_factory=dict)
    result: str = ""
    content: str = ""
    timestamp: float = 0.0  # absolute wall clock (s since epoch) when the harness recorded it
    meta: dict[str, Any] | None = None  # e.g. compaction start/end timestamps


@dataclass
class ToolCall:
    index: int  # order among tool calls in the task (0-based)
    tool: str
    args: dict[str, Any]
    author: str
    elapsed: float
    result: str | None = None  # raw response text (JSON string) if matched
    timestamp: float = 0.0

    def result_json(self) -> dict[str, Any] | None:
        if self.result is None:
            return None
        try:
            v = json.loads(self.result)
        except (ValueError, TypeError):
            return None
        # ADK sometimes wraps tool output as {"result": "<json string>"}
        if isinstance(v, dict) and set(v) == {"result"} and isinstance(v["result"], str):
            try:
                inner = json.loads(v["result"])
                if isinstance(inner, dict):
                    return inner
            except ValueError:
                pass
        return v if isinstance(v, dict) else None


def _atif_events(data: dict[str, Any]) -> list[Event]:
    evs: list[Event] = []
    for step in data.get("steps", []):
        source = step.get("source", "agent")
        msg = step.get("message")
        extra = step.get("extra") or {}
        author = extra.get("author", source)
        etype = extra.get("event_type")
        el = float(extra.get("elapsed_s", 0.0) or 0.0)
        ts = float(extra.get("timestamp", 0.0) or 0.0)
        if msg is not None or (etype is not None and not step.get("tool_calls") and not step.get("observation") and etype != "usage"):
            if not etype:
                etype = {"system": "system_instruction", "user": "task_prompt"}.get(source, "text")
            evs.append(Event(etype, author, el, content=str(msg) if msg is not None else "", timestamp=ts,
                             meta=extra.get("metadata")))
        for tc in step.get("tool_calls", []) or []:
            tx = tc.get("extra") or {}
            evs.append(Event("tool_call", tx.get("author", author), float(tx.get("elapsed_s", el) or 0.0),
                             tool=tc.get("function_name", ""), args=tc.get("arguments") or {},
                             timestamp=float(tx.get("timestamp", ts) or 0.0)))
        obs = step.get("observation")
        if obs:
            ox = obs.get("extra") or {}
            content = obs.get("content", "")
            if not isinstance(content, str):
                content = json.dumps(content, default=str)
            evs.append(Event("tool_response", ox.get("author") or "system", float(ox.get("elapsed_s", el) or 0.0),
                             tool=ox.get("tool_name", ""), result=content, timestamp=float(ox.get("timestamp", ts) or 0.0)))
    return evs


def _legacy_events(data: dict[str, Any]) -> list[Event]:
    return [
        Event(e.get("type") or "custom", e.get("author") or "", float(e.get("elapsed_s") or 0.0),
              tool=e.get("tool") or "", args=e.get("args") or {}, result=e.get("result") or "", content=e.get("content") or "",
              timestamp=float(e.get("timestamp") or 0.0), meta=e.get("metadata"))
        for e in data.get("entries", [])
    ]


def parse_trace(data: dict[str, Any]) -> list[Event]:
    if str(data.get("schema_version", "")).startswith("ATIF"):
        return _atif_events(data)
    return _legacy_events(data)


def load_trace(path: str | Path) -> list[Event]:
    with open(path, encoding="utf-8") as f:
        return parse_trace(json.load(f))


def pair_tool_calls(events: list[Event]) -> list[ToolCall]:
    """Match each tool_response to the earliest unanswered tool_call with the same tool name."""
    calls: list[ToolCall] = []
    pending: dict[str, list[ToolCall]] = {}
    for ev in events:
        if ev.type == "tool_call":
            c = ToolCall(len(calls), ev.tool, ev.args or {}, ev.author, ev.elapsed, timestamp=ev.timestamp)
            calls.append(c)
            pending.setdefault(ev.tool, []).append(c)
        elif ev.type == "tool_response":
            q = pending.get(ev.tool)
            if q:
                q.pop(0).result = ev.result
    return calls

In [ ]:
%%writefile /kaggle/working/pilot/common/metrics.py
"""Per-task and aggregate metrics shared by all pilots.

Common metric set (identical definitions in every pilot):
  resolved          scorer verdict (Phase 2 pytest + JUnit validation) AND non-empty patch, as in the
                    scorer, which never verifies an empty prediction (resolved_local = raw harness flag)
  wall_s            TaskResult.duration_seconds (setup + agent + verification)
  agent_s           last event timestamp in the trace (time inside the agent loop)
  tool_calls        tool_call events in the ROOT trace. Calls made inside an AgentTool sub-agent
                    (e.g. code_analyzer_agent) run in a separate ADK Runner and are NOT in the
                    trace; only the AgentTool call itself is (verified in google-adk 1.36.1
                    tools/agent_tool.py; pilot A found this first). Localization metrics are
                    therefore lower bounds when the agent delegates reading to the sub-agent.
  budget_tool_calls harness-counted calls (TaskResult.tool_calls; budget-gated tools only)
  repeat_share      share of tool calls identical (same tool + canonical JSON args) to an
                    earlier call in the same task
  malformed_share   malformed / (tool_calls + unparsed_tool_calls); malformed = unknown tool,
                    missing/unexpected arguments, or argument validation errors; unparsed =
                    turns cut off inside an unclosed <|tool_call> (harness nudge type 1)
  failed_edits      edit_file / write_file calls that returned status=error
  skill_calls       calls to ADK skill tools (list/search/load_skill, load_skill_resource,
                    run_skill_script)
Optional localization metrics (need gold locations, see common.tasks.gold_locations):
  viewed_gold_file, first_gold_view_call, edited_gold_file, edited_gold_function
"""
from __future__ import annotations

import json
import re
import statistics
from typing import Any, Iterable

from .diffs import parse_unified_diff
from .trace import Event, ToolCall, pair_tool_calls

SKILL_TOOLS = {"list_skills", "search_skills", "load_skill", "load_skill_resource", "run_skill_script"}
EDIT_TOOLS = {"edit_file", "write_file"}
_MALFORMED_TYPES = {"TypeError", "ValidationError", "KeyError", "JSONDecodeError", "ArgumentError"}
_MALFORMED_PATTERNS = (
    re.compile(r"Tool '.*' not found", re.S),
    re.compile(r"mandatory input parameters are not present"),
    re.compile(r"unexpected keyword argument"),
    re.compile(r"missing \d+ required (positional|keyword)"),
    re.compile(r"cannot be greater than end_line"),
)
_UNPARSED_NUDGE = "tool call finished closing"


def canonical_call(c: ToolCall) -> str:
    return c.tool + "\x00" + json.dumps(c.args, sort_keys=True, default=str)


def _error_info(c: ToolCall) -> tuple[bool, str, str]:
    r = c.result_json()
    if r is None:
        return False, "", ""
    if r.get("status") == "error":
        return True, str(r.get("error_type", "")), str(r.get("error_message") or r.get("error") or "")
    return False, "", ""


def is_malformed(c: ToolCall) -> bool:
    is_err, etype, msg = _error_info(c)
    if not is_err:
        return False
    if etype in _MALFORMED_TYPES:
        return True
    return any(p.search(msg) for p in _MALFORMED_PATTERNS)


def call_touches_path(c: ToolCall, path: str) -> bool:
    """True if a tool call reads/views `path` (read_file, or a shell command naming it)."""
    if c.tool == "read_file":
        fp = str(c.args.get("filepath", "")).lstrip("/")
        fp = fp[len("workspace/"):] if fp.startswith("workspace/") else fp
        return fp == path or fp.endswith("/" + path)
    if c.tool == "run_command":
        return path in str(c.args.get("command", ""))
    return False


def task_metrics(
    result: dict[str, Any],
    events: list[Event],
    gold: dict[str, Any] | None = None,
    agent_patch_scopes: Iterable[str] | None = None,
) -> dict[str, Any]:
    """Compute the common metric row for one task (root-agent calls only, see module doc).

    result: a TaskResult row (task_results.jsonl line or TaskResult.model_dump()).
    events: parsed trace (common.trace.load_trace / parse_trace).
    gold:   optional GoldLocations.to_dict() for localization metrics.
    agent_patch_scopes: optional dotted names of scopes the agent's patch changed.
    The returned row holds numbers/flags only (no patch, no trace text).
    """
    calls = pair_tool_calls(events)
    n = len(calls)
    seen: set[str] = set()
    repeats = 0
    for c in calls:
        k = canonical_call(c)
        if k in seen:
            repeats += 1
        seen.add(k)
    unparsed = sum(1 for e in events if e.type == "continuation_nudge" and _UNPARSED_NUDGE in e.content)
    nudges = sum(1 for e in events if e.type == "continuation_nudge")
    malformed = sum(1 for c in calls if is_malformed(c))
    edit_calls = [c for c in calls if c.tool in EDIT_TOOLS]
    failed_edits = sum(1 for c in edit_calls if _error_info(c)[0])
    by_tool: dict[str, int] = {}
    for c in calls:
        by_tool[c.tool] = by_tool.get(c.tool, 0) + 1
    agent_s = max((e.elapsed for e in events), default=0.0)
    patch = result.get("agent_patch") or ""
    patch_files = [fd.path for fd in parse_unified_diff(patch)] if patch else []

    row: dict[str, Any] = {
        "instance_id": result.get("task_id") or result.get("instance_id"),
        # The scorer only verifies NON-EMPTY predictions (empty -> "NO_PATCH", unresolved), while the local
        # swegemma Evaluator also verifies empty patches; a task whose tests already pass at base_commit then
        # counts as resolved locally. `resolved` follows the scorer; `resolved_local` is the raw harness flag.
        "resolved": bool(result.get("resolved")) and bool((result.get("agent_patch") or "").strip()),
        "resolved_local": bool(result.get("resolved")),
        "error": (result.get("error_message") or "")[:120] or None,
        "wall_s": round(float(result.get("duration_seconds") or 0.0), 1),
        "agent_s": round(agent_s, 1),
        "tool_calls": n,
        "budget_tool_calls": int(result.get("tool_calls") or 0),
        "llm_calls": int(result.get("total_llm_calls") or 0),
        "total_tokens": int(result.get("total_tokens") or 0),
        "repeat_calls": repeats,
        "repeat_share": round(repeats / n, 4) if n else 0.0,
        "malformed_calls": malformed,
        "unparsed_tool_calls": unparsed,
        "malformed_share": round((malformed + unparsed) / (n + unparsed), 4) if (n + unparsed) else 0.0,
        "edit_calls": len(edit_calls),
        "failed_edits": failed_edits,
        "skill_calls": sum(1 for c in calls if c.tool in SKILL_TOOLS),
        "nudges": nudges,
        "submitted": any(c.tool == "submit_patch" for c in calls),
        "patch_nonempty": bool(patch.strip()),
        "patch_files": len(patch_files),
        "calls_by_tool": by_tool,
    }
    if gold is not None:
        gfiles = list(gold.get("src_files") or [])
        first = next((c.index for c in calls if any(call_touches_path(c, g) for g in gfiles)), None)
        row["viewed_gold_file"] = first is not None
        row["first_gold_view_call"] = first
        row["edited_gold_file"] = bool(set(patch_files) & set(gfiles))
        row["edited_only_gold_files"] = bool(patch_files) and set(patch_files) <= set(gfiles)
        if agent_patch_scopes is not None:
            gfun = {f["dotted"] for f in gold.get("functions") or []}
            row["edited_gold_function"] = bool(gfun & set(agent_patch_scopes)) if gfun else None
    return row


def _mean(xs: list[float]) -> float | None:
    return round(statistics.fmean(xs), 3) if xs else None


def _median(xs: list[float]) -> float | None:
    return round(statistics.median(xs), 3) if xs else None


def aggregate(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """Aggregate common metrics over a list of task rows (one condition/run)."""
    n = len(rows)
    tot_calls = sum(r["tool_calls"] for r in rows)
    tot_unparsed = sum(r["unparsed_tool_calls"] for r in rows)
    out: dict[str, Any] = {
        "n_tasks": n,
        "resolved": sum(r["resolved"] for r in rows),
        "resolve_rate": round(sum(r["resolved"] for r in rows) / n, 4) if n else None,
        "wall_s_mean": _mean([r["wall_s"] for r in rows]),
        "wall_s_median": _median([r["wall_s"] for r in rows]),
        "agent_s_mean": _mean([r["agent_s"] for r in rows]),
        "tool_calls_mean": _mean([r["tool_calls"] for r in rows]),
        # pooled shares (calls across all tasks) and per-task means
        "repeat_share_pooled": round(sum(r["repeat_calls"] for r in rows) / tot_calls, 4) if tot_calls else 0.0,
        "malformed_share_pooled": round((sum(r["malformed_calls"] for r in rows) + tot_unparsed) / (tot_calls + tot_unparsed), 4)
        if (tot_calls + tot_unparsed) else 0.0,
        "repeat_share_task_mean": _mean([r["repeat_share"] for r in rows]),
        "malformed_share_task_mean": _mean([r["malformed_share"] for r in rows]),
        "failed_edits_total": sum(r["failed_edits"] for r in rows),
        "edit_calls_total": sum(r["edit_calls"] for r in rows),
        "skill_calls_total": sum(r["skill_calls"] for r in rows),
        "submitted_share": round(sum(r["submitted"] for r in rows) / n, 4) if n else None,
        "patch_nonempty_share": round(sum(r["patch_nonempty"] for r in rows) / n, 4) if n else None,
        "errors": sum(1 for r in rows if r.get("error")),
    }
    for k in ("viewed_gold_file", "edited_gold_file", "edited_only_gold_files", "edited_gold_function"):
        vals = [r[k] for r in rows if r.get(k) is not None]
        if vals:
            out[f"{k}_share"] = round(sum(vals) / len(vals), 4)
    fg = [r["first_gold_view_call"] for r in rows if r.get("first_gold_view_call") is not None]
    if fg:
        out["first_gold_view_call_median"] = _median(fg)
    return out


def paired_flips(a: list[dict[str, Any]], b: list[dict[str, Any]]) -> dict[str, Any]:
    """Paired comparison of two runs over the same tasks (b vs a).

    Returns discordant counts and an exact two-sided McNemar (binomial) p-value.
    """
    ra = {r["instance_id"]: r["resolved"] for r in a}
    rb = {r["instance_id"]: r["resolved"] for r in b}
    common = sorted(set(ra) & set(rb))
    gain = sum(1 for t in common if rb[t] and not ra[t])
    loss = sum(1 for t in common if ra[t] and not rb[t])
    m = gain + loss
    if m == 0:
        p = 1.0
    else:
        from math import comb
        k = min(gain, loss)
        p = min(1.0, 2 * sum(comb(m, i) for i in range(k + 1)) / 2 ** m)
    return {"n_common": len(common), "b_only": gain, "a_only": loss, "both": sum(1 for t in common if ra[t] and rb[t]),
            "delta": (sum(rb[t] for t in common) - sum(ra[t] for t in common)), "mcnemar_exact_p": round(p, 4)}

In [ ]:
%%writefile /kaggle/working/pilot/common/runtime_probes.py
"""Runtime instrumentation for swegemma / google-adk runs (generic; counts only, no content saved).

Why: in google-adk 1.36.1 token-threshold compaction runs as a request processor before a model
call (flows/llm_flows/compaction.py) and appends its event to the session WITHOUT yielding it, so
the harness SessionTrace (fed by yielded events) is expected to miss it. The summarizer
(apps/llm_event_summarizer.py) formats only `part.text` of the compacted events, so function
calls/responses (incl. load_skill outputs) are dropped, and the initial task prompt is compacted
like any other event (all but the last `event_retention_size` events).

This module measures that directly:
  REQUESTS     one record per LLM request: messages by role, tool messages, whether the task
               prompt ("Problem Statement:") and each tracked marker string are still present
  COMPACTIONS  one record per summarizer call: events compacted, function calls / responses
               dropped, text vs non-text chars, whether the task prompt was compacted, which
               tracked markers were present before and survive in the summary
Task attribution: a ContextVar set around Evaluator.evaluate_task.

Usage: install(); set MARKERS[task_id] = [strings to track]; run; read REQUESTS / COMPACTIONS.
"""
from __future__ import annotations

import contextvars
import json
import os
import time
from collections import Counter
from typing import Any

CURRENT_TASK: contextvars.ContextVar[str | None] = contextvars.ContextVar("pilot_current_task", default=None)
REQUESTS: list[dict[str, Any]] = []
COMPACTIONS: list[dict[str, Any]] = []
MARKERS: dict[str, list[str]] = {}
TASK_PROMPT_MARKER = "Problem Statement:"
AGENT_REQUEST_TIMEOUT_S = 900
SKILL_RESPONSE_TOOLS = {"load_skill", "load_skill_resource"}
_installed = False


def _msg_text(m: Any) -> str:
    c = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
    if c is None:
        return ""
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return " ".join(str(x.get("text", "")) if isinstance(x, dict) else str(x) for x in c)
    return str(c)


def _get(m: Any, k: str) -> Any:
    return m.get(k) if isinstance(m, dict) else getattr(m, k, None)


def request_shape(messages: list[Any], tools: Any) -> dict[str, Any]:
    tid = CURRENT_TASK.get()
    texts = [_msg_text(m) for m in messages]
    roles = Counter(str(_get(m, "role")) for m in messages)
    marks = MARKERS.get(tid or "", [])
    return {
        "task": tid, "t": time.time(), "n_messages": len(messages), "roles": dict(roles),
        "n_tool_messages": roles.get("tool", 0),
        "n_assistant_tool_calls": sum(len(_get(m, "tool_calls") or []) for m in messages),
        "has_tools": bool(tools), "chars": sum(len(x) for x in texts),
        "has_task_prompt": any(TASK_PROMPT_MARKER in x for x in texts),
        "markers_present": [i for i, s in enumerate(marks) if any(s in x for x in texts)],
    }


def compaction_record(events: list[Any], result: Any) -> dict[str, Any]:
    tid = CURRENT_TASK.get()
    marks = MARKERS.get(tid or "", [])
    n_fc = n_fr = n_skill = 0
    text_chars = nontext_chars = 0
    task_prompt = False
    raw_blobs: list[str] = []
    for e in events:
        parts = getattr(getattr(e, "content", None), "parts", None) or []
        for p in parts:
            if getattr(p, "text", None):
                text_chars += len(p.text)
                raw_blobs.append(p.text)
                if getattr(e, "author", "") == "user" and TASK_PROMPT_MARKER in p.text:
                    task_prompt = True
            fc = getattr(p, "function_call", None)
            if fc is not None:
                n_fc += 1
                blob = json.dumps(dict(fc.args or {}), default=str)
                nontext_chars += len(blob)
                raw_blobs.append(blob)
            fr = getattr(p, "function_response", None)
            if fr is not None:
                n_fr += 1
                if getattr(fr, "name", "") in SKILL_RESPONSE_TOOLS:
                    n_skill += 1
                blob = json.dumps(getattr(fr, "response", None), default=str)
                nontext_chars += len(blob)
                raw_blobs.append(blob)
    summary = ""
    comp = getattr(getattr(result, "actions", None), "compaction", None)
    cc = getattr(comp, "compacted_content", None)
    for p in getattr(cc, "parts", None) or []:
        if getattr(p, "text", None):
            summary += p.text
    raw = "\n".join(raw_blobs)
    before = [i for i, s in enumerate(marks) if s in raw]
    return {
        "task": tid, "t": time.time(), "n_events": len(events), "function_calls_dropped": n_fc,
        "function_responses_dropped": n_fr, "skill_responses_dropped": n_skill,
        "text_chars": text_chars, "nontext_chars": nontext_chars, "task_prompt_compacted": task_prompt,
        "summary_chars": len(summary), "summary_has_task_prompt_marker": TASK_PROMPT_MARKER in summary,
        "markers_before": before, "markers_in_summary": [i for i in before if marks[i] in summary],
        "produced": result is not None,
    }


def install() -> None:
    """Patch (idempotently) the ADK LLM client, the ADK summarizer and swegemma's evaluate_task."""
    global _installed
    if _installed:
        return
    from google.adk.apps import llm_event_summarizer as les
    from google.adk.models import lite_llm
    from swegemma import evaluate as sg_eval

    import litellm
    from common.server_watchdog import tracked

    # per-request timeout for agent requests (LiteLLM's default is 6000 s). 900 s = the per-task time budget:
    # a single request longer than the whole task budget is useless anyway.
    timeout_s = float(os.environ.get("PILOT_AGENT_REQUEST_TIMEOUT", AGENT_REQUEST_TIMEOUT_S))
    litellm.request_timeout = timeout_s
    orig_ac = lite_llm.LiteLLMClient.acompletion

    async def acompletion(self, model, messages, tools, **kwargs):
        try:
            REQUESTS.append(request_shape(messages, tools))
        except Exception:  # noqa: BLE001 - instrumentation must never break a run
            pass
        kwargs.setdefault("timeout", timeout_s)
        with tracked():  # counts in-flight requests for the watchdog; fails fast after a freeze
            return await orig_ac(self, model, messages, tools, **kwargs)

    orig_sum = les.LlmEventSummarizer.maybe_summarize_events

    async def maybe_summarize_events(self, *, events):
        res = await orig_sum(self, events=events)
        try:
            COMPACTIONS.append(compaction_record(events, res))
        except Exception:  # noqa: BLE001
            pass
        return res

    orig_eval = sg_eval.Evaluator.evaluate_task

    async def evaluate_task(self, *args, **kwargs):
        from common.server_watchdog import ServerFrozen, STATE, TRIPPED
        if TRIPPED.is_set():  # server frozen: do not even set up the task's sandbox
            raise ServerFrozen(f"inference server frozen ({STATE['reason']}); task skipped")
        task = kwargs.get("task", args[0] if args else None)
        tok = CURRENT_TASK.set(getattr(task, "instance_id", None))
        try:
            return await orig_eval(self, *args, **kwargs)
        finally:
            CURRENT_TASK.reset(tok)

    lite_llm.LiteLLMClient.acompletion = acompletion
    les.LlmEventSummarizer.maybe_summarize_events = maybe_summarize_events
    sg_eval.Evaluator.evaluate_task = evaluate_task
    _installed = True


def reset() -> None:
    REQUESTS.clear()
    COMPACTIONS.clear()


def task_compaction_stats(task_id: str, tool_call_timestamps: list[float]) -> dict[str, Any]:
    """Per-task summary: how often and when compaction fired, what it dropped, what survived."""
    comps = [c for c in COMPACTIONS if c["task"] == task_id and c["produced"]]
    reqs = [r for r in REQUESTS if r["task"] == task_id and r["has_tools"]]
    first_t = comps[0]["t"] if comps else None
    after = [r for r in reqs if first_t is not None and r["t"] > first_t]
    marks_before = sum(len(c["markers_before"]) for c in comps)
    return {
        "compactions": len(comps),
        "first_compaction_at_call": sum(1 for ts in tool_call_timestamps if first_t is not None and ts < first_t) if comps else None,
        "function_calls_dropped": sum(c["function_calls_dropped"] for c in comps),
        "function_responses_dropped": sum(c["function_responses_dropped"] for c in comps),
        "skill_responses_dropped": sum(c["skill_responses_dropped"] for c in comps),
        "nontext_share_dropped": round(sum(c["nontext_chars"] for c in comps) / max(1, sum(c["nontext_chars"] + c["text_chars"] for c in comps)), 4) if comps else None,
        "task_prompt_compacted": any(c["task_prompt_compacted"] for c in comps),
        "marker_survival": round(sum(len(c["markers_in_summary"]) for c in comps) / marks_before, 4) if marks_before else None,
        "requests": len(reqs),
        "requests_after_first_compaction": len(after),
        "after_with_task_prompt": sum(r["has_task_prompt"] for r in after),
        "max_tool_messages_before": max((r["n_tool_messages"] for r in reqs if first_t is None or r["t"] <= first_t), default=0),
        "max_tool_messages_after": max((r["n_tool_messages"] for r in after), default=None),
    }

In [ ]:
%%writefile /kaggle/working/pilot/common/server_watchdog.py
"""Freeze protection for a local OpenAI-compatible inference server (vLLM) during GPU notebooks.

Why: a main-track forum report says the gemma4 tool parser can loop forever on an array tool argument
(in the harness tool set the only one is `get_code_subgraph(nodes: list[str])`) and freeze the server
for the rest of the run. Without protection a freeze burns L4x4 time until the session limit: LiteLLM's
default request timeout is 6000 s, and adk-eval-core's ModelRetryPlugin retries timeouts up to 5 times.

Three layers, used by every GPU notebook:
  1. per-request timeouts (set by the callers: runtime_probes for agent requests, probe_runner for probes);
  2. `Watchdog`: a thread polling GET /health; after `max_fail` consecutive failures, or when requests
     are in flight but none has finished for `stall_s`, it trips: it sets TRIPPED and kills the server;
  3. `tracked()`: wraps every request; once TRIPPED is set, new and retried requests fail immediately with
     ServerFrozen (an adk-eval-core EvalCoreError when available, which its retry plugin does not retry).
The notebook loop then restarts the server for the next unit of work, and stops the session after a
second freeze.
"""
from __future__ import annotations

import contextlib
import os
import threading
import time
import urllib.request
from typing import Any, Callable

try:  # non-retryable for adk-eval-core's ModelRetryPlugin
    from adk_eval_core.errors import EvalCoreError as _Base
except Exception:  # noqa: BLE001
    _Base = RuntimeError


class ServerFrozen(_Base):
    """The inference server stopped responding and was killed by the watchdog."""


TRIPPED = threading.Event()
STATE: dict[str, Any] = {"reason": None, "t": None}
_LOCK = threading.Lock()
TRACK: dict[str, Any] = {"inflight": 0, "started": 0, "done": 0, "last_done": time.time()}
_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def reset() -> None:
    TRIPPED.clear()
    STATE.update({"reason": None, "t": None})
    with _LOCK:
        TRACK.update({"inflight": 0, "started": 0, "done": 0, "last_done": time.time()})


@contextlib.contextmanager
def tracked():
    """Wrap one request to the inference server: counts it, and fails fast after a freeze."""
    if TRIPPED.is_set():
        raise ServerFrozen(f"inference server frozen ({STATE['reason']}); request not sent")
    with _LOCK:
        TRACK["inflight"] += 1
        TRACK["started"] += 1
    try:
        yield
    except Exception as e:  # noqa: BLE001
        if TRIPPED.is_set() and not isinstance(e, ServerFrozen):
            raise ServerFrozen(f"inference server frozen ({STATE['reason']}): {type(e).__name__}") from e
        raise
    finally:
        with _LOCK:
            TRACK["inflight"] -= 1
            TRACK["done"] += 1
            TRACK["last_done"] = time.time()


def _env(name: str, default: float) -> float:
    return float(os.environ.get(name, default))


class Watchdog(threading.Thread):
    def __init__(self, root_url: str, kill: Callable[[], None], interval: float | None = None,
                 health_timeout: float | None = None, max_fail: int | None = None, stall_s: float | None = None,
                 log: Callable[[str], None] = print):
        super().__init__(daemon=True)
        self.root = root_url.rstrip("/")
        self.root = self.root[:-3] if self.root.endswith("/v1") else self.root
        self.kill = kill
        # defaults: a freeze is detected within ~90 s; overridable for dry runs
        self.interval = interval if interval is not None else _env("PILOT_WD_INTERVAL", 30)
        self.health_timeout = health_timeout if health_timeout is not None else _env("PILOT_WD_HEALTH_TIMEOUT", 10)
        self.max_fail = int(max_fail if max_fail is not None else _env("PILOT_WD_MAX_FAIL", 3))
        self.stall_s = stall_s if stall_s is not None else _env("PILOT_WD_STALL_S", 1200)
        self.log = log
        self._halt = threading.Event()  # not `_stop`: that name is a threading.Thread internal
        self.checks = 0

    def healthy(self) -> bool:
        try:
            with _OPENER.open(f"{self.root}/health", timeout=self.health_timeout) as r:
                return 200 <= r.status < 300
        except Exception:  # noqa: BLE001
            return False

    def run(self) -> None:
        fails = 0
        while not self._halt.wait(self.interval):
            self.checks += 1
            fails = 0 if self.healthy() else fails + 1
            with _LOCK:
                stalled = TRACK["inflight"] > 0 and time.time() - TRACK["last_done"] > self.stall_s
            if fails >= self.max_fail or stalled:
                self.trip("health check failed %d times" % fails if fails >= self.max_fail
                          else "no request finished for %.0f s" % self.stall_s)
                return

    def trip(self, reason: str) -> None:
        STATE.update({"reason": reason, "t": time.time()})
        TRIPPED.set()
        self.log(f"WATCHDOG: inference server considered frozen ({reason}); killing it")
        killer = threading.Thread(target=self._kill, daemon=True)
        killer.start()
        killer.join(60)

    def _kill(self) -> None:
        try:
            self.kill()
        except Exception as e:  # noqa: BLE001
            self.log(f"WATCHDOG: kill raised {type(e).__name__}: {e}")

    def stop(self) -> None:
        self._halt.set()


def state() -> dict[str, Any]:
    with _LOCK:
        return {"tripped": TRIPPED.is_set(), **STATE, **{k: v for k, v in TRACK.items() if k != "last_done"}}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/__init__.py
# (empty module)

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/hints.py
"""Oracle-hint rendering for Pilot C (locations and test names only, never patch content).

Conditions (nested for localization, orthogonal for tests):
  O0      no hint
  O-file  gold source file(s)
  O-func  gold file(s) + enclosing function/class names (with def line)
  O-test  scorer-required test names + their test file(s)
  O-all   O-func + O-test

Hints are appended to problem_statement in a LOCAL copy of the tasks used only
by the local evaluation runner. They must never be placed in a submission.
"""
from __future__ import annotations

import copy

CONDITIONS = ("O0", "O-file", "O-func", "O-test", "O-all")
_HEADER = "## Location hints (verified ground truth for this task)"


def _file_lines(g: dict) -> list[str]:
    out = []
    new = set(g.get("new_src_files") or [])
    for f in g.get("src_files") or []:
        out.append(f"- The fix changes `{f}`" + (" (a new file that does not exist yet)." if f in new else "."))
    if not out:
        out.append("- The fix does not change any existing non-test Python source file.")
    return out


def _func_lines(g: dict) -> list[str]:
    out = []
    by_file: dict[str, list[dict]] = {}
    for fn in g.get("functions") or []:
        by_file.setdefault(fn["file"], []).append(fn)
    for f in g.get("src_files") or []:
        for fn in by_file.get(f, []):
            what = "class body of" if fn["kind"] == "class" else "function"
            out.append(f"- In `{f}`: {what} `{fn['dotted']}` (defined at line {fn['def_line']}).")
        if f in (g.get("module_level_files") or []):
            out.append(f"- In `{f}`: module-level code (imports, constants, or new top-level definitions).")
    return out


def _test_lines(g: dict) -> list[str]:
    names = g.get("test_names") or []
    files = g.get("test_files") or []
    new = set(g.get("new_test_files") or [])
    out = []
    for f in files:
        status = "this test file does not exist yet; it will be added when your fix is verified" if f in new \
            else "this test file exists; the listed tests may be new or changed"
        out.append(f"- Verification runs `{f}` ({status}).")
    if names:
        out.append("- Tests that must pass: " + ", ".join(f"`{n}`" for n in names) + ".")
    return out


def render_hint(g: dict, condition: str) -> str:
    if condition not in CONDITIONS:
        raise ValueError(condition)
    if condition == "O0":
        return ""
    lines: list[str] = []
    if condition in ("O-file", "O-func", "O-all"):
        lines += _file_lines(g)
    if condition in ("O-func", "O-all"):
        lines += _func_lines(g)
    if condition in ("O-test", "O-all"):
        lines += _test_lines(g)
    return "\n".join([_HEADER, *lines])


def rewrite_task(task: dict, g: dict, condition: str) -> dict:
    """Return a copy of `task` with the hint appended to problem_statement."""
    t = copy.deepcopy(task)
    hint = render_hint(g, condition)
    if hint:
        t["problem_statement"] = t["problem_statement"].rstrip() + "\n\n" + hint + "\n"
    return t

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/select_tasks.py
"""Select the pilot tasks (instance IDs only are written; safe to commit).

Rule (fixed before any agent run):
  eligible = exactly 1 changed source file that already exists at base_commit,
             1-2 enclosing functions/classes (function-level oracle well defined),
             gold patch <= 40 changed lines, >= 1 scorer-required test name,
             repo in {fastapi, rich, requests},
             and the GOLD patch resolves the task in the sandbox (gold check JSON, if given)
  pilot15  = stratified random sample: fastapi 7, rich 6, requests 2 (seeded)
  reserve  = next 4 per repo from the same shuffled order (replace tasks whose gold
             patch fails the CPU gold check)
  smoke3   = first pilot task of each repo (used for the smoke test and size timing)

usage: python -I pilot_c/select_tasks.py <gold_all.json> <out task_ids.json> [gold_check.json ...]
"""
import json
import random
import sys

SEED = 20261031
QUOTA = {"fastapi/fastapi": 7, "Textualize/rich": 6, "psf/requests": 2}
RESERVE = 4


def eligible(g):
    return (len(g["src_files"]) == 1 and not g["new_src_files"] and 1 <= len(g["functions"]) <= 2
            and g["gold_added"] + g["gold_removed"] <= 40 and len(g["test_names"]) >= 1 and g["repo"] in QUOTA)


def main():
    G = json.load(open(sys.argv[1]))
    checks = {}
    for path in sys.argv[3:]:
        for r in json.load(open(path))["rows"]:
            checks[r["instance_id"]] = bool(r["gold_resolved"])
    if checks:
        G = {i: g for i, g in G.items() if checks.get(i) is True}
    rng = random.Random(SEED)
    pilot, reserve, smoke, pool_sizes = [], [], [], {}
    for repo, k in QUOTA.items():
        pool = sorted(i for i, g in G.items() if g["repo"] == repo and eligible(g))
        pool_sizes[repo] = len(pool)
        rng.shuffle(pool)
        pilot += pool[:k]
        reserve += pool[k:k + RESERVE]
        smoke.append(pool[0])
    out = {"seed": SEED, "gold_check_sources": sys.argv[3:], "n_gold_checked": len(checks), "rule": __doc__.split("Rule (fixed before any agent run):")[1].split("usage:")[0].strip(),
           "eligible_pool_sizes": pool_sizes, "pilot15": pilot, "reserve": reserve, "smoke3": smoke}
    json.dump(out, open(sys.argv[2], "w"), indent=1)
    print(json.dumps({k: out[k] for k in ("eligible_pool_sizes", "smoke3")}), len(pilot), len(reserve))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/kaggle_runner.py
"""Kaggle-side runner for Pilot C (imported by the notebooks; runs only on Kaggle).

Reproduces the official starter's evaluation path (swegemma Evaluator, subprocess
sandbox, vLLM with max_model_len=32768, tool/reasoning parser gemma4, the same
compaction and context-cache settings as the scorer per HARNESS_README §7.2) and
changes only: model alias, per-task budget (eval_config.yaml), task copies with
oracle hints, results dir, concurrency.

Outputs:
  /kaggle/working/private/...         raw runs (patches, traces) - NEVER commit
  /kaggle/working/C0x_<what>.json      numbers only - commit to results/ (names: notebooks/README.md)
"""
from __future__ import annotations

import asyncio
import concurrent.futures
import datetime as _dt
import glob
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))

from common.metrics import aggregate, paired_flips, task_metrics  # noqa: E402
from common.tasks import diff_scopes, gold_locations, load_tasks  # noqa: E402
from common import runtime_probes as rp  # noqa: E402
from common.trace import load_trace  # noqa: E402
from pilot_c.hints import _HEADER as HINT_HEADER, rewrite_task  # noqa: E402

WORK = Path(os.environ.get("PILOT_WORK_DIR", "/kaggle/working"))
PRIVATE = WORK / "private"
ALLOWED = ("gemma-4-31b-it-qat-w4a16-ct", "gemma-4-12b-it-qat-w4a16-ct",
           "gemma-4-e4b-it-qat-w4a16-ct", "gemma-4-e2b-it-qat-w4a16-ct")
SHORT = {m: m.split("-")[2] for m in ALLOWED}  # 31b / 12b / e4b / e2b
MODEL_VLLM_EXTRA_ARGS = {
    "gemma-4-12b-it-qat-w4a16-ct": ("--hf-overrides", '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}'),
}
GPU_ONLY_WHEELS = ("vllm", "flashinfer", "bitsandbytes", "xgrammar", "nvidia_", "quack_kernels", "apache_tvm_ffi")


def _first_dir(*cands: str) -> Path:
    for c in cands:
        for p in sorted(glob.glob(c)):
            if Path(p).is_dir():
                return Path(p)
    raise FileNotFoundError(f"none of {cands} found")


def data_dir() -> Path:
    if os.environ.get("PILOT_DATA_DIR"):  # local testing only
        return Path(os.environ["PILOT_DATA_DIR"])
    return _first_dir("/kaggle/input/competitions/gemma-4-developer-agent", "/kaggle/input/gemma-4-developer-agent")


def wheelhouse_dir() -> Path:
    return _first_dir("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse",
                      "/kaggle/input/gemma-4-developer-agent-wheelhouse")


def set_env() -> None:
    """Same environment variables as the official starter notebook."""
    env = {
        "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1", "TRANSFORMERS_OFFLINE": "1",
        "VLLM_NO_USAGE_STATS": "1", "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
        "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1", "NCCL_DEBUG": "WARN", "LITELLM_LOCAL_MODEL_COST_MAP": "True",
        "LITELLM_TELEMETRY": "False", "LITELLM_LOG": "WARNING", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "OTEL_SDK_DISABLED": "true", "OTEL_PYTHON_DISABLED": "true",
    }
    os.environ.update(env)
    os.environ.setdefault("KAGGLE_SANDBOX_DIR", str(data_dir() / "sandbox"))
    os.environ.setdefault("OPENAI_API_KEY", "EMPTY")
    os.environ.setdefault("LOCAL_API_KEY", "EMPTY")


EXPECTED_PYTHON = (3, 12)  # the official starter's pinned Kaggle environment; the wheelhouse is built for cp312
# Wheelhouse wheels the CPU notebooks need (gold check, test verification). Found by running that code path and
# recording every imported distribution: of 90, only these 5 come from the wheelhouse; all are pure Python
# (py3-none-any). Everything else is provided by the Kaggle image. The other 36 wheels serve the model (vLLM stack).
CPU_WHEELS = ("swegemma", "adk_eval_core", "adk_submission", "google_adk", "google_genai")


def _dist_of(wheel_name: str) -> str:
    return re.sub(r"[-_.]+", "_", wheel_name.split("-")[0]).lower()


def _wheel_supported(wheel_name: str) -> bool:
    """True if the wheel's python tag fits this interpreter (pure-python, abi3 or the exact cpXY)."""
    parts = wheel_name[:-4].split("-")
    py, abi = parts[-3], parts[-2]
    here = f"cp{sys.version_info.major}{sys.version_info.minor}"
    if py.startswith("py") or "none" in abi and not py.startswith("cp"):
        return True
    if abi == "abi3":
        return any(int(t[3:] or 0) <= sys.version_info.minor for t in py.split(".") if t.startswith("cp3"))
    return here in py.split(".")


def install_wheelhouse(gpu: bool = True, only: tuple[str, ...] | None = None) -> None:
    """Starter-identical wheel install. gpu=False skips GPU-only wheels; `only` restricts to those distributions
    (the CPU notebooks pass CPU_WHEELS). Refuses, before calling pip, any wheel built for another Python."""
    import importlib
    for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth", "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for pth in glob.glob(pat):
            try:
                os.unlink(pth)
            except OSError:
                pass
    tmp = Path("/tmp/wheelhouse")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)
    for w in wheelhouse_dir().glob("*.whl"):
        if "cutlass" in w.name.lower() or (not gpu and w.name.startswith(GPU_ONLY_WHEELS)):
            continue
        if only is not None and _dist_of(w.name) not in only:
            continue
        name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
        t = tmp / name
        if not t.exists():
            try:
                os.symlink(w, t)
            except OSError:
                shutil.copyfile(w, t)
    wheels = sorted(str(w) for w in tmp.glob("*.whl"))
    if only is not None:
        missing = sorted(set(only) - {_dist_of(Path(w).name) for w in wheels})
        if missing:
            raise RuntimeError(f"wheelhouse is missing {missing}; is the dataset metric/gemma-4-developer-agent-wheelhouse attached?")
    bad = [Path(w).name for w in wheels if not _wheel_supported(Path(w).name)]
    if bad:
        raise RuntimeError(f"Python {sys.version.split()[0]} cannot install {bad[:3]}... (wheelhouse is built for "
                           f"cp{EXPECTED_PYTHON[0]}{EXPECTED_PYTHON[1]}). Run this notebook in the official starter's "
                           "pinned environment: see the first cell of the notebook.")
    print(f"installing {len(wheels)} wheels: {', '.join(_dist_of(Path(w).name) for w in wheels)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels], check=True)
    importlib.invalidate_caches()


def quiet_logs() -> None:
    import logging
    for name in ("LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "litellm", "httpx", "httpcore", "openai", "urllib3",
                 "opentelemetry", "opentelemetry.context"):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if "litellm" in name.lower():
            lg.propagate = False
            lg.handlers.clear()


def run_sync(fn, *args, **kwargs):
    """Run an async callable from a notebook (same helper as the starter)."""
    coro_fn = lambda: fn(*args, **kwargs)  # noqa: E731
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())


def env_info() -> dict[str, Any]:
    info: dict[str, Any] = {"python": platform.python_version(), "nproc": os.cpu_count()}
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        info["gpus"] = out
    except Exception:  # noqa: BLE001
        info["gpus"] = []
    try:
        import importlib.metadata as md
        info["versions"] = {p: md.version(p) for p in ("swegemma", "adk-eval-core", "adk-submission", "google-adk", "vllm")
                            if _has_dist(md, p)}
    except Exception:  # noqa: BLE001
        pass
    return info


def _has_dist(md, p: str) -> bool:
    try:
        md.version(p)
        return True
    except Exception:  # noqa: BLE001
        return False


# --------------------------------------------------------------------------- tasks & hints

def build_gold(task_ids: list[str]) -> dict[str, dict]:
    d = data_dir()
    tasks = {t["instance_id"]: t for t in load_tasks(d / "tasks.jsonl")}
    out = {}
    for iid in task_ids:
        out[iid] = gold_locations(tasks[iid], d / "snapshots" / f"{iid}.tgz").to_dict()
    return out


def write_task_copy(task_ids: list[str], gold: dict[str, dict], condition: str) -> Path:
    """Local hinted copy of tasks.jsonl for one condition (kept under PRIVATE)."""
    d = data_dir()
    tasks = {t["instance_id"]: t for t in load_tasks(d / "tasks.jsonl")}
    PRIVATE.mkdir(parents=True, exist_ok=True)
    path = PRIVATE / f"tasks_{condition}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for iid in task_ids:
            f.write(json.dumps(rewrite_task(tasks[iid], gold[iid], condition)) + "\n")
    return path


# --------------------------------------------------------------------------- agent & server

def prepare_agent_dir(model: str, budget: dict[str, Any]) -> Path:
    """Official sample_submission with the model alias swapped and the sample's dummy LoRA
    adapters removed for EVERY size (starter logic for non-31B; for 31B too, so all sizes run the
    plain base model), and eval_config.yaml replaced by the pilot budget."""
    assert model in ALLOWED, model
    src = data_dir() / "sample_submission"
    dst = WORK / f"agent_{SHORT[model]}"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    shutil.rmtree(dst / "adapters", ignore_errors=True)
    for y in [dst / "agent.yaml", *sorted((dst / "sub_agents").glob("*.yaml"))]:
        if y.is_file():
            txt = y.read_text(encoding="utf-8")
            txt = re.sub(r"^(\s*model:\s*).*$", rf"\g<1>{model}", txt, flags=re.M)
            txt = re.sub(r"^\s*adapter:\s*.*\n?", "", txt, flags=re.M)
            y.write_text(txt, encoding="utf-8")
    lines = ["evaluation:"] + [f"  {k}: {v}" for k, v in budget.items()]
    (dst / "eval_config.yaml").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return dst


def describe_agent(agent_dir: Path) -> dict[str, Any]:
    import yaml
    a = yaml.safe_load((agent_dir / "agent.yaml").read_text().replace("!include", "#include")) or {}
    samp = yaml.safe_load((agent_dir / "configs" / "sampling.yaml").read_text())
    tools = [t if isinstance(t, str) else "agent_tool:" + str(t.get("agent_tool", {}).get("config_path")) for t in a.get("tools", [])]
    return {"model": a.get("model"), "tools": tools, "sampling": samp,
            "eval_config": yaml.safe_load((agent_dir / "eval_config.yaml").read_text())}


class GpuMemoryNotReleased(RuntimeError):
    """GPU memory did not come back after a server was stopped; starting another server would fail."""


SERVER_LOG_DIR = PRIVATE / "server_logs"
LAST_SERVER_ERRORS: list[str] = []


def _own_process_group() -> None:
    """Make adk_submission start every inference server in its own session / process group.

    VllmServer.start() uses a plain Popen, so the server shares the notebook kernel's process group and
    VllmServer.stop() kills only the API-server PID: vLLM's EngineCore and its tensor-parallel worker
    processes survive and keep ~20 GiB on every GPU (Kaggle C03 run, 2026-10-31)."""
    import types
    import adk_submission.server as ads
    if getattr(ads.subprocess, "_pilot_own_group", False):
        return
    real = subprocess

    def popen(*args, **kwargs):
        kwargs.setdefault("start_new_session", True)
        return real.Popen(*args, **kwargs)

    shim = types.SimpleNamespace(**{n: getattr(real, n) for n in dir(real) if not n.startswith("__")})
    shim.Popen = popen
    shim._pilot_own_group = True
    ads.subprocess = shim


def gpu_memory() -> list[tuple[int, int]] | None:
    """(free MiB, total MiB) per GPU from nvidia-smi, or None when there is no GPU (dry runs)."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.free,memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=30)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return None
    if out.returncode != 0 or not out.stdout.strip():
        return None
    return [tuple(int(float(x)) for x in line.split(",")[:2]) for line in out.stdout.strip().splitlines()]


def wait_gpu_free(need_frac: float, timeout_s: float = 300, poll_s: float = 5) -> dict[str, Any]:
    """Block until every GPU has at least need_frac of its memory free; raise GpuMemoryNotReleased on timeout."""
    t0 = time.time()
    while True:
        mem = gpu_memory()
        if mem is None:
            return {"checked": False}
        if all(free >= need_frac * total for free, total in mem):
            return {"checked": True, "waited_s": round(time.time() - t0, 1), "free_gib": [round(f / 1024, 2) for f, _ in mem]}
        if time.time() - t0 > timeout_s:
            raise GpuMemoryNotReleased(
                f"GPU memory not released {timeout_s:.0f}s after stopping the previous server: free "
                f"{[round(f / 1024, 2) for f, _ in mem]} GiB of {[round(t / 1024, 2) for _, t in mem]} GiB; the next "
                f"server needs {need_frac:.0%} free on every GPU. Stopping instead of starting another server.")
        time.sleep(poll_s)


def _save_server_log(server, tag: str) -> list[str]:
    """Keep the server log (VllmServer.stop() deletes it) under private/, return its error lines."""
    path = getattr(server, "log_path", None)
    if not path or not os.path.exists(path):
        return []
    try:
        lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError:
        return []
    SERVER_LOG_DIR.mkdir(parents=True, exist_ok=True)
    (SERVER_LOG_DIR / f"{tag}_{int(time.time())}.log").write_text("\n".join(lines[-2000:]), encoding="utf-8")
    errs = [l[:300] for l in lines if re.search(r"ERROR|Traceback|Error:|out of memory|OutOfMemory", l)]
    return errs[-20:]


def error_kinds(lines: list[str]) -> list[str]:
    """Exception class names only (safe to commit; the lines themselves stay private)."""
    return sorted({m for l in lines for m in re.findall(r"\b([A-Z][A-Za-z]+(?:Error|Exception))\b", l)})


def stop_server(server, tag: str = "server", timeout: float = 15.0) -> dict[str, Any]:
    """Stop an inference server and everything it spawned: SIGTERM, then SIGKILL, to its whole process group,
    plus any descendants recorded before the kill. Saves the server log first."""
    import signal
    global LAST_SERVER_ERRORS
    LAST_SERVER_ERRORS = _save_server_log(server, tag)
    info: dict[str, Any] = {"group_killed": False, "survivors": 0, "error_kinds": error_kinds(LAST_SERVER_ERRORS)}
    proc = getattr(server, "process", None)
    if proc is None:
        safe_stop(server)
        return info
    pid = proc.pid
    try:
        import psutil
        desc = psutil.Process(pid).children(recursive=True)
    except Exception:  # noqa: BLE001
        psutil, desc = None, []
    try:
        pgid = os.getpgid(pid)
    except ProcessLookupError:
        pgid = None
    own = pgid is not None and pgid != os.getpgrp()
    try:
        if own:
            os.killpg(pgid, signal.SIGTERM)
        else:
            proc.terminate()
        try:
            proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            pass
        if own:
            try:
                os.killpg(pgid, signal.SIGKILL)
                info["group_killed"] = True
            except ProcessLookupError:
                pass
        elif proc.poll() is None:
            proc.kill()
        for d in desc:
            try:
                d.kill()
            except Exception:  # noqa: BLE001
                pass
        if psutil is not None and desc:
            _, alive = psutil.wait_procs(desc, timeout=15)
            info["survivors"] = len(alive)
        try:
            proc.wait(timeout=10)
        except Exception:  # noqa: BLE001
            pass
    finally:
        safe_stop(server)
    info["descendants"] = len(desc)
    return info


def start_server(agent_dir: Path, gpu_memory_utilization: float = 0.90, extra_args: tuple[str, ...] = ()):
    """Starter-identical vLLM launch (except optional memory settings for the probe), in its own process group.
    Waits until every GPU is free first. Returns (server, models, adapters, startup_seconds)."""
    if os.environ.get("PILOT_MOCK_LLM"):  # local papermill dry runs only (no GPU, scripted mock LLM)
        return _start_mock_server(agent_dir)
    import litellm
    import torch
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from adk_submission.discovery import discover_declared_models
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

    _own_process_group()
    litellm.drop_params = True
    declared = discover_declared_models(agent_dir)
    assert len(declared) == 1, declared
    model = next(iter(declared))
    assert model in ALLOWED, model
    model_path = Path(f"/kaggle/input/models/google/gemma-4/other/{model}/2")
    if not model_path.exists():
        model_path = _first_dir(f"/kaggle/input/models/google/gemma-4/*/{model}/*", f"/kaggle/input/gemma-4/*/{model}/*")
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    amap = getattr(adapters, "adapters", None) or {}
    ranks = [r for info in amap.values() if (r := getattr(info, "rank", None)) is not None and r > 0]
    n = torch.cuda.device_count() if torch.cuda.is_available() else 1
    tp = 4 if n >= 4 else (2 if n >= 2 else 1)
    waited = wait_gpu_free(gpu_memory_utilization + 0.02)
    cfg = VllmConfig(
        model=str(model_path), port=8000, host="127.0.0.1", tool_call_parser="gemma4", reasoning_parser="gemma4",
        max_model_len=32768, dtype="bfloat16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "auto",
        gpu_memory_utilization=gpu_memory_utilization, enable_auto_tool_choice=True, enable_lora=bool(amap),
        max_loras=max(1, len(amap)), max_lora_rank=max(ranks) if ranks else 128, scheduler_reserve_full_isl=False,
        tensor_parallel_size=tp, startup_timeout=60 * 20,
        extra_args=tuple(MODEL_VLLM_EXTRA_ARGS.get(model, ())) + tuple(extra_args),
    )
    t0 = time.perf_counter()
    server = VllmServer(cfg, adapter_manifest=adapters)
    try:
        server.start()
    except Exception:
        stop_server(server, tag=f"{SHORT[model]}_failed_start")  # also kills workers of a half-started server
        print("server log errors:", *LAST_SERVER_ERRORS[-8:], sep="\n  ")
        raise
    startup = time.perf_counter() - t0
    models = server.create_model_registry(aliases=[model], model_prefix="openai/", api_key=os.environ.get("LOCAL_API_KEY", "EMPTY"))
    print(f"vLLM up for {model} in {startup:.0f}s (tp={tp}, gpu_memory_utilization={gpu_memory_utilization}, "
          f"extra={list(extra_args)}, waited for GPU memory: {waited})")
    return server, models, adapters, startup


def _start_mock_server(agent_dir: Path):
    """Dry-run stand-in for vLLM: the scripted OpenAI-compatible mock from tests/ (PILOT_REPO)."""
    import litellm
    from adk_submission import discover_adapters
    from adk_submission.registry import ModelRegistry
    from adk_submission.resolvers.generation import install_litellm_reasoning_patch
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
    sys.path.insert(0, os.environ["PILOT_REPO"])
    from tests.mock_openai_server import kill, make_script, serve

    litellm.drop_params = True
    install_litellm_reasoning_patch()
    model = describe_agent(agent_dir)["model"]
    if os.environ.get("PILOT_MOCK_FAIL_START") == SHORT.get(model):  # dry-run test of a size whose server fails
        raise RuntimeError("mock: inference server exited prematurely")
    srv = serve(make_script("README.md"), port=int(os.environ.get("PILOT_MOCK_PORT", "8001")))
    reg = ModelRegistry()
    reg.register(model, LiteLlm(model="openai/mock", api_base=f"http://127.0.0.1:{srv.server_address[1]}/v1", api_key="EMPTY"))
    ad = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

    class _S:
        base_url = f"http://127.0.0.1:{srv.server_address[1]}/v1"

        def stop(self):
            kill(srv)
    return _S(), reg, ad, 0.0


def server_root(server) -> str:
    """http://host:port of a started server (VllmServer.base_url or the dry-run mock's), without /v1."""
    base = str(getattr(server, "base_url", None) or "http://127.0.0.1:8000").rstrip("/")
    return base[:-3] if base.endswith("/v1") else base


def safe_stop(server) -> None:
    """Stop a server that may already have been killed by the watchdog."""
    try:
        server.stop()
    except Exception as e:  # noqa: BLE001
        print(f"server.stop(): {type(e).__name__}: {e}")


# --------------------------------------------------------------------------- evaluation

def make_eval_config(agent_dir: Path, models, adapters, tasks_path: Path, task_ids: list[str], results_dir: Path,
                     concurrency: int, sandbox: str = "subprocess"):
    import yaml
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import EvalConfig, build_submission_limits

    raw = yaml.safe_load((agent_dir / "eval_config.yaml").read_text()) or {}
    ev = raw.get("evaluation", raw)
    turns = ev.get("max_turns", ev.get("max_llm_calls"))
    limits, gen = build_submission_limits()
    d = data_dir()
    return EvalConfig(
        tasks_path=tasks_path, snapshots_dir=d / "snapshots", results_dir=results_dir, submission_dir=agent_dir,
        models=models, sandbox=sandbox,
        timeout_seconds=int(ev.get("timeout_seconds", 300)),
        verification_timeout_seconds=int(ev.get("verification_timeout_seconds", 300)),
        max_time_minutes=float(ev.get("max_time_minutes", 60.0)), max_tool_calls=int(ev.get("max_tool_calls", 100)),
        max_turns=int(turns) if turns is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336,
                                                        event_retention_size=5),
        graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), wheels_dir=d / "wheels",
        verbose=False, task_ids=list(task_ids), concurrency=concurrency, display_mode="quiet",
    )


def _result_dict(r) -> dict[str, Any]:
    try:
        d = r.model_dump(exclude={"trace"})
    except Exception:  # noqa: BLE001
        d = dict(r.__dict__)
        d.pop("trace", None)
    d.setdefault("instance_id", getattr(r, "instance_id", None) or d.get("task_id"))
    return d


def collect_rows(results, results_dir: Path, gold: dict[str, dict]) -> list[dict[str, Any]]:
    d = data_dir()
    rows = []
    for r in results:
        rd = _result_dict(r)
        iid = rd.get("instance_id") or rd.get("task_id")
        tpath = rd.get("trace_path") or str(results_dir / "traces" / f"trace_{iid}.json")
        events = load_trace(tpath) if tpath and Path(tpath).exists() else []
        patch = rd.get("agent_patch") or ""
        scopes = diff_scopes(patch, d / "snapshots" / f"{iid}.tgz") if patch.strip() else []
        row = task_metrics(rd, events, gold=gold.get(iid), agent_patch_scopes=scopes)
        row["trace_found"] = bool(events)
        row["trace_compaction_entries"] = sum(1 for e in events if e.type == "compaction")
        row.update({f"cmp_{k}": v for k, v in rp.task_compaction_stats(
            iid, [e.timestamp for e in events if e.type == "tool_call"]).items()})
        rows.append(row)
    return rows


def compaction_summary(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """Run-level answer to: how often/when does compaction fire, what does it drop, is it in the trace?"""
    fired = [r for r in rows if r.get("cmp_compactions")]
    firsts = sorted(r["cmp_first_compaction_at_call"] for r in fired if r.get("cmp_first_compaction_at_call") is not None)
    surv = [r["cmp_marker_survival"] for r in fired if r.get("cmp_marker_survival") is not None]
    return {
        "tasks": len(rows), "tasks_with_compaction": len(fired),
        "compactions_total": sum(r.get("cmp_compactions") or 0 for r in rows),
        "first_compaction_at_call_median": firsts[len(firsts) // 2] if firsts else None,
        "first_compaction_at_call_values": firsts,
        "function_calls_dropped": sum(r.get("cmp_function_calls_dropped") or 0 for r in rows),
        "function_responses_dropped": sum(r.get("cmp_function_responses_dropped") or 0 for r in rows),
        "skill_responses_dropped": sum(r.get("cmp_skill_responses_dropped") or 0 for r in rows),
        "task_prompt_compacted_tasks": sum(bool(r.get("cmp_task_prompt_compacted")) for r in fired),
        "requests_after_compaction_with_task_prompt": sum(r.get("cmp_after_with_task_prompt") or 0 for r in fired),
        "requests_after_compaction": sum(r.get("cmp_requests_after_first_compaction") or 0 for r in fired),
        "marker_survival_mean": round(sum(surv) / len(surv), 4) if surv else None,
        "trace_compaction_entries_total": sum(r.get("trace_compaction_entries") or 0 for r in rows),
        "markers": "per task: hint header, gold file path(s), gold function short names",
    }


def _markers_for(g: dict) -> list[str]:
    out = [HINT_HEADER] + list(g.get("src_files") or [])
    out += [f["qualname"].split(".")[-1] for f in g.get("functions") or []]
    return list(dict.fromkeys(out))


def run_condition(agent_dir: Path, models, adapters, task_ids: list[str], gold: dict[str, dict], condition: str,
                  replicate: int, concurrency: int) -> dict[str, Any]:
    from swegemma.evaluate import Evaluator
    model_short = agent_dir.name.replace("agent_", "")
    tasks_path = write_task_copy(task_ids, gold, condition)
    rdir = PRIVATE / "runs" / model_short / f"{condition}_r{replicate}"
    if rdir.exists():
        shutil.rmtree(rdir)
    rdir.mkdir(parents=True)
    cfg = make_eval_config(agent_dir, models, adapters, tasks_path, task_ids, rdir, concurrency)
    rp.install()
    rp.reset()
    for iid in task_ids:
        rp.MARKERS[iid] = _markers_for(gold.get(iid) or {})
    t0 = time.perf_counter()
    res = run_sync(Evaluator(cfg).run)
    batch = time.perf_counter() - t0
    rows = collect_rows(res.task_results, rdir, gold)
    agg = aggregate(rows)
    agg["compaction"] = compaction_summary(rows)
    print(f"[{model_short} {condition} r{replicate}] resolved {agg['resolved']}/{agg['n_tasks']} "
          f"batch {batch/60:.1f} min, mean wall {agg['wall_s_mean']}s, calls {agg['tool_calls_mean']}, "
          f"compactions {agg['compaction']['compactions_total']} (in trace: {agg['compaction']['trace_compaction_entries_total']})")
    return {"model": model_short, "condition": condition, "replicate": replicate, "concurrency": concurrency,
            "batch_wall_s": round(batch, 1), "aggregate": agg, "rows": rows}


def save_summary(path: Path, summary: dict[str, Any]) -> None:
    runs = summary.get("runs", [])
    comps = []
    base = {(r["model"]): r for r in runs if r["condition"] == "O0" and r["replicate"] == 1}
    for r in runs:
        b = base.get(r["model"])
        if b is not None and r is not b:
            comps.append({"model": r["model"], "a": "O0_r1", "b": f"{r['condition']}_r{r['replicate']}",
                          **paired_flips(b["rows"], r["rows"])})
    summary["paired_vs_O0_r1"] = comps
    summary["saved_utc"] = _dt.datetime.utcnow().isoformat(timespec="seconds")
    path.write_text(json.dumps(summary, indent=1))
    print(f"saved {path}")


# --------------------------------------------------------------------------- CPU gold check

def gold_check(task_ids: list[str], out_path: Path, timeout_s: int = 300, baseline: bool = True) -> dict[str, Any]:
    """CPU-only: apply each task's GOLD patch in the scorer's Phase-2 sandbox and record resolved; also run the
    same verification with NO patch (baseline). valid = gold passes AND no-patch fails. Numbers only."""
    from adk_submission.registry import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.deduplication import resolve_task_snapshot_paths
    from swegemma.evaluate import Evaluator
    from swegemma.harness.verification import verify_task
    from swegemma.models import load_tasks as sg_load

    d = data_dir()
    PRIVATE.mkdir(parents=True, exist_ok=True)
    cfg = EvalConfig(tasks_path=d / "tasks.jsonl", snapshots_dir=d / "snapshots", results_dir=PRIVATE / "gold_check",
                     submission_dir=d / "sample_submission", models=ModelRegistry(), sandbox="subprocess",
                     timeout_seconds=timeout_s, verification_timeout_seconds=timeout_s, wheels_dir=d / "wheels",
                     graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), display_mode="quiet")
    ev = Evaluator(cfg)
    tasks = {t.instance_id: t for t in sg_load(d / "tasks.jsonl")}
    rows = []
    for i, iid in enumerate(task_ids, 1):
        t = tasks[iid]
        snap, base, patch = resolve_task_snapshot_paths(d / "snapshots", iid, t.repo)
        t0 = time.perf_counter()
        out_text = ""
        try:
            r = run_sync(verify_task, ev.docker, cfg, t, snap, base_snapshot_path=base, patch_path=patch,
                         agent_patch=t.patch, agent_error=None, trace=None, start_time=t0)
            ok, err = bool(r.resolved), (r.error or "")[:160] or None
            out_text = r.test_output or ""
        except Exception as e:  # noqa: BLE001
            ok, err = False, f"{type(e).__name__}: {e}"[:160]
        dt = time.perf_counter() - t0
        base_ok = None
        if baseline:  # the same tests with NO patch must fail, else any harmless edit "solves" the task
            try:
                rb = run_sync(verify_task, ev.docker, cfg, t, snap, base_snapshot_path=base, patch_path=patch,
                              agent_patch="", agent_error=None, trace=None, start_time=time.perf_counter())
                base_ok = bool(rb.resolved)
            except Exception:  # noqa: BLE001
                base_ok = None
        cls, mods = classify_gold_failure(ok, err, out_text)
        rows.append({"instance_id": iid, "repo": t.repo, "gold_resolved": ok, "baseline_resolved": base_ok,
                     "valid": bool(ok and base_ok is False), "verify_s": round(dt, 1), "error": err,
                     "failure_class": cls, "missing_modules": mods})  # for missing_fixture: fixture names
        print(f"[{i}/{len(task_ids)}] {iid} gold={ok} no_patch={base_ok} {cls} {mods or ''} {dt:.0f}s {err or ''}")
        out_path.write_text(json.dumps({"rows": rows, "summary": gold_check_summary(rows), "env": env_info()}, indent=1))
    return {"rows": rows, "summary": gold_check_summary(rows)}


_MISSING_MOD = re.compile(r"No module named '([A-Za-z0-9_\.]+)'")
_FIXTURE = re.compile(r"fixture '([A-Za-z0-9_]+)' not found|recursive dependency involving fixture '([A-Za-z0-9_]+)'")


def classify_gold_failure(ok: bool, err: str | None, test_output: str) -> tuple[str, list[str]]:
    """Why a gold patch failed (module names / fixture names only; test output is not stored)."""
    if ok:
        return "resolved", []
    mods = sorted({m.split(".")[0] for m in _MISSING_MOD.findall(test_output or "")})
    if mods:
        return "missing_module", mods
    fx = sorted({a or b for a, b in _FIXTURE.findall(test_output or "")})
    if fx:
        return "missing_fixture", fx
    e = err or ""
    if "Failed to apply" in e:
        return "patch_apply_failed", []
    if "TimeoutExceeded" in e or "timed out" in e.lower():
        return "timeout", []
    if e:
        return "harness_error", []
    return "test_failure", []


def gold_check_summary(rows: list[dict[str, Any]]) -> dict[str, Any]:
    from collections import Counter
    by_repo: dict[str, dict[str, int]] = {}
    for r in rows:
        d = by_repo.setdefault(r.get("repo", "?"), {"checked": 0, "gold_resolved": 0})
        d["checked"] += 1
        d["gold_resolved"] += int(r["gold_resolved"])
        d["valid"] = d.get("valid", 0) + int(bool(r.get("valid")))
    mods = Counter(m for r in rows if r["failure_class"] == "missing_module" for m in r.get("missing_modules") or [])
    fixtures = Counter(m for r in rows if r["failure_class"] == "missing_fixture" for m in r.get("missing_modules") or [])
    fastapi_inline_or_dirty = sum(1 for r in rows if r.get("repo") == "fastapi/fastapi" and r["failure_class"] == "missing_module"
                                  and {"inline_snapshot", "dirty_equals"} & set(r.get("missing_modules") or []))
    return {"checked": len(rows), "gold_resolved": sum(r["gold_resolved"] for r in rows),
            "baseline_resolved_without_patch": sum(1 for r in rows if r.get("baseline_resolved")),
            "valid": sum(1 for r in rows if r.get("valid")), "by_repo": by_repo,
            "failure_classes": dict(Counter(r["failure_class"] for r in rows)), "missing_modules": dict(mods),
            "missing_fixtures": dict(fixtures),
            "fastapi_failing_on_inline_snapshot_or_dirty_equals": fastapi_inline_or_dirty}


def select_from_gold_check(gold_all: dict[str, dict], check_rows: list[dict[str, Any]], seed: int = 20261031) -> dict[str, Any]:
    """Pre-registered order: per repo, eligible VALID tasks (gold passes and no-patch fails HERE), seeded shuffle.
    smoke3 = first of each repo. IDs only."""
    import random
    from pilot_c.select_tasks import QUOTA, eligible
    ok = {r["instance_id"] for r in check_rows if r.get("valid", r["gold_resolved"])}
    rng = random.Random(seed)
    order: dict[str, list[str]] = {}
    for repo in QUOTA:
        pool = sorted(i for i, g in gold_all.items() if g["repo"] == repo and eligible(g) and i in ok)
        rng.shuffle(pool)
        order[repo] = pool
    smoke = [v[0] for v in order.values() if v]
    # fill rule (pre-registered): if a repo has no valid task, take the next valid tasks round-robin
    # from the other repos' orders, so the timing set always has 3 tasks when 3 exist
    k = 1
    while len(smoke) < 3 and any(len(v) > k for v in order.values()):
        for v in order.values():
            if len(v) > k and len(smoke) < 3:
                smoke.append(v[k])
        k += 1
    return {"seed": seed, "eligible_valid_in_order": order, "smoke3": smoke, "valid_all": sorted(ok)}


def find_input_json(name: str) -> Path:
    """Locate an attached notebook output (e.g. C00_gold_check.json from notebooks/C00_gold_check__CPU.ipynb)
    anywhere under /kaggle/input, independent of the slug Kaggle gives the attached notebook."""
    root = os.environ.get("PILOT_INPUT_ROOT", "/kaggle/input")
    hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
    if not hits:
        producer = {"C00_gold_check.json": "notebooks/C00_gold_check__CPU.ipynb",
                    "probe_records.jsonl": "notebooks/C03_probe__L4x4.ipynb"}.get(name, "the notebook that writes it")
        raise FileNotFoundError(f"{name} not found under {root}: attach the output of {producer} as an input first "
                                "(see notebooks/README.md)")
    return Path(hits[0])

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/task_ids.json
{
 "status": "Candidates drawn with the pre-registered rule, pre-filtered by a LOCAL gold check. The pilot notebook re-checks gold on Kaggle and fills each repo quota in this order.",
 "seed": 20261031,
 "rule": "eligible = exactly 1 changed source file that already exists at base_commit,\n             1-2 enclosing functions/classes (function-level oracle well defined),\n             gold patch <= 40 changed lines, >= 1 scorer-required test name,\n             repo in {fastapi, rich, requests},\n             and the GOLD patch resolves the task in the sandbox (gold check JSON, if given)\n  pilot15  = stratified random sample: fastapi 7, rich 6, requests 2 (seeded)\n  reserve  = next 4 per repo from the same shuffled order (replace tasks whose gold\n             patch fails the CPU gold check)\n  smoke3   = first pilot task of each repo (used for the smoke test and size timing)",
 "quota": {
  "fastapi/fastapi": 7,
  "Textualize/rich": 6,
  "psf/requests": 2
 },
 "candidates_in_order": {
  "fastapi/fastapi": [
   "fastapi_9753",
   "fastapi_15589",
   "fastapi_5077",
   "fastapi_14479",
   "fastapi_14463",
   "fastapi_14301",
   "fastapi_5624",
   "fastapi_13920",
   "fastapi_14794",
   "fastapi_9425",
   "fastapi_14873"
  ],
  "Textualize/rich": [
   "rich_3518",
   "rich_3944",
   "rich_4075",
   "rich_4076",
   "rich_3480",
   "rich_3067",
   "rich_3894",
   "rich_3882",
   "rich_4077",
   "rich_3470"
  ],
  "psf/requests": [
   "requests_7309",
   "requests_7427",
   "requests_7315"
  ]
 },
 "pilot15": [
  "fastapi_9753",
  "fastapi_15589",
  "fastapi_5077",
  "fastapi_14479",
  "fastapi_14463",
  "fastapi_14301",
  "fastapi_5624",
  "rich_3518",
  "rich_3944",
  "rich_4075",
  "rich_4076",
  "rich_3480",
  "rich_3067",
  "requests_7309",
  "requests_7427"
 ],
 "reserve": [
  "fastapi_13920",
  "fastapi_14794",
  "fastapi_9425",
  "fastapi_14873",
  "rich_3894",
  "rich_3882",
  "rich_4077",
  "rich_3470",
  "requests_7315"
 ],
 "eligible_pool_sizes_after_local_gold_check": {
  "fastapi/fastapi": 18,
  "Textualize/rich": 22,
  "psf/requests": 3
 },
 "smoke3_gpu": [
  "fastapi_15588",
  "rich_3454",
  "requests_7309"
 ]
}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/probe.py
"""Information-ladder probe: per-task records, canonical actions, rung texts, controls, classification.

Pure CPU code (no harness imports). Implements docs/PLAN.md §3-§5 and the Level B record of
docs/PROBE_INPUT.md (paper/main). Outputs derived from task data stay private (on Kaggle).
"""
from __future__ import annotations

import json
import re
import tarfile
from pathlib import Path
from typing import Any

from common.diffs import FileDiff, is_test_path, parse_unified_diff, required_test_names
from common.locate import collect_scopes, module_path, report_scope
from pilot_c.hints import _HEADER as LOC_HEADER

MAX_READ_LINES = 150
MAX_READ_CHARS = 10000
TEST_SOURCE_MAX_LINES = 60

LADDER = ("L0", "L1", "L2", "L3", "L4", "L5")
FIND_CONDS = ("L0", "L1", "L2", "T", "W", "G-json", "G-edges", "G-adj", "X-file", "X-func", "X-graph")
USE_CONDS = ("L0", "L1", "L2", "L3", "L4", "L5", "T", "W", "G-json", "G-edges", "G-adj",
             "X-file", "X-func", "X-graph", "X-test", "X-words")
D2_FIND = ("L0", "L1", "L2", "G-json", "G-edges", "X-func")
D2_USE = ("L0", "L2", "L3", "L4", "L5", "T", "W", "G-json", "X-func", "X-test")
GRAPH_CONDS = {"G-json": "json", "G-edges": "edges", "G-adj": "adj", "X-graph": "json"}
WORDS_CONDS = {"L4", "L5", "W", "X-words"}


# --------------------------------------------------------------------------- snapshot access

def snapshot_scan(tgz: str | Path, wanted: set[str]) -> tuple[dict[str, str], list[str]]:
    """One pass over the archive: text of `wanted` files + list of all non-test .py paths."""
    texts: dict[str, str] = {}
    py: list[str] = []
    with tarfile.open(tgz, mode="r|gz") as tf:
        for m in tf:
            name = m.name[2:] if m.name.startswith("./") else m.name
            if not m.isfile() or name.startswith(".git/"):
                continue
            if name.endswith(".py") and not is_test_path(name):
                py.append(name)
            if name in wanted:
                fh = tf.extractfile(m)
                if fh is not None:
                    texts[name] = fh.read().decode("utf-8", errors="replace")
    return texts, sorted(py)


def harness_read_output(filepath: str, text: str, start: int, end: int) -> str:
    """Exact swegemma 0.2.11 read_file response (tools/workspace.py) for a line range."""
    lines = text.splitlines()
    total = len(lines)
    s, e = max(start, 1), min(end, total)
    trunc = False
    if e - s + 1 > MAX_READ_LINES:
        e, trunc = s + MAX_READ_LINES - 1, True
    snippet = "\n".join(lines[s - 1:e])
    if len(snippet) > MAX_READ_CHARS:
        snippet, trunc = snippet[:MAX_READ_CHARS], True
    return json.dumps({"status": "ok", "filepath": filepath, "content": snippet, "start_line": s, "end_line": e,
                       "total_lines": total, "is_truncated": trunc}, ensure_ascii=False)


# --------------------------------------------------------------------------- canonical edit

def _new_region(lines: list[str], fd: FileDiff, a: int, b: int) -> list[str]:
    removed = set(fd.removed_old_lines)
    ins: dict[int, list[str]] = {}
    for x in fd.insertions:
        ins.setdefault(x.after_old, []).extend(x.lines)
    out: list[str] = []
    for ln in range(a, b + 1):
        if ln not in removed:
            out.append(lines[ln - 1])
        out.extend(ins.get(ln, []))
    return out


def canonical_edit(base: str, fd: FileDiff, fs: int, fe: int) -> dict[str, Any] | None:
    """old/new strings for the gold change inside lines [fs, fe] of the base file (PLAN §4.3)."""
    lines = base.splitlines()
    rem = [ln for ln in fd.removed_old_lines if fs <= ln <= fe]
    ins = [x for x in fd.insertions if fs <= x.after_old <= fe and any(l.strip() for l in x.lines)]
    if not rem and not ins:
        return None
    pts = rem + [x.after_old for x in ins]
    a, b = max(1, min(pts)), min(len(lines), max(pts))
    for _ in range(12):
        old = "\n".join(lines[a - 1:b])
        if old.strip() and base.count(old) == 1:
            break
        if b < min(len(lines), fe + 3):
            b += 1
        elif a > max(1, fs - 3):
            a -= 1
        else:
            return None
    else:
        return None
    new = "\n".join(_new_region(lines, fd, a, b))
    while not new.strip() and (a > 1 or b < len(lines)):  # pure deletion: keep a neighbouring anchor line
        if a > 1:
            a -= 1
        else:
            b += 1
        old = "\n".join(lines[a - 1:b])
        new = "\n".join(_new_region(lines, fd, a, b))
    if new == old or base.count(old) != 1:
        return None
    changed = sorted(set(rem) | {x.after_old for x in ins})
    outside = sum(1 for ln in fd.removed_old_lines if not fs <= ln <= fe) + \
        sum(1 for x in fd.insertions if not fs <= x.after_old <= fe and any(l.strip() for l in x.lines))
    return {"old_string": old, "new_string": new, "old_start": a, "old_end": b,
            "changed_lines": [min(changed), max(changed)], "outside_edits": outside}


def apply_edit(base: str, old: str, new: str) -> tuple[str | None, str]:
    """Apply an edit_file call. Uses the harness's own 4-tier apply_replacement (adk-eval-core) when it is
    installed (Kaggle, dry runs); otherwise a local exact + whitespace-flexible approximation."""
    try:
        from adk_eval_core.editing.edit import apply_replacement
    except ImportError:
        apply_replacement = None
    if apply_replacement is not None:
        r = apply_replacement(base, old, new)
        ok = not getattr(r, "error_message", None) and getattr(r, "occurrences", 0) == 1
        return (r.new_content if ok else None), (getattr(r, "strategy", "harness") if ok else "none")
    if old and base.count(old) == 1:
        return base.replace(old, new, 1), "exact"
    bl = base.splitlines()
    ol = [x.strip() for x in old.splitlines()]
    if not ol:
        return None, "none"
    hits = [i for i in range(len(bl) - len(ol) + 1) if [x.strip() for x in bl[i:i + len(ol)]] == ol]
    if len(hits) != 1:
        return None, "none"
    i = hits[0]
    ind = len(bl[i]) - len(bl[i].lstrip())
    nl = new.splitlines()
    nind = (len(nl[0]) - len(nl[0].lstrip())) if nl else 0
    shifted = [(" " * max(0, ind + len(x) - len(x.lstrip()) - nind) + x.lstrip()) if x.strip() else x for x in nl]
    return "\n".join(bl[:i] + shifted + bl[i + len(ol):]) + ("\n" if base.endswith("\n") else ""), "flexible"


def _norm(text: str) -> str:
    return "\n".join(l.rstrip() for l in text.strip().splitlines())


# --------------------------------------------------------------------------- test source / diff

def test_source(test_patch: str, names: list[str] | None = None, max_lines: int = TEST_SOURCE_MAX_LINES) -> str:
    """The reproduction-test rung: test_patch as it reads AFTER applying it (context + added lines,
    no removed lines), per file and hunk, capped at `max_lines`. Uniform for new test functions and
    for changes to test data/parametrization (where the scorer's test name may not enclose the change)."""
    out: list[str] = []
    for fd_block in re.split(r"(?m)^(?=--- )", test_patch):
        if not fd_block.strip():
            continue
        m = re.search(r"(?m)^\+\+\+ b/(.+)$", fd_block)
        out.append(f"# {m.group(1) if m else '?'}")
        for hunk in re.split(r"(?m)^(?=@@ )", fd_block)[1:]:
            body = [l[1:] for l in hunk.splitlines()[1:] if l[:1] in ("+", " ")]
            out += body + ["# ..."]
    return "\n".join(out[:max_lines]).rstrip()


def file_diff_text(patch: str, path: str) -> str:
    for block in re.split(r"(?m)^(?=--- )", patch):
        if block.startswith(f"--- a/{path}\n") or f"\n+++ b/{path}\n" in block[:400]:
            return block.rstrip("\n")
    return ""


# --------------------------------------------------------------------------- Level B record

def build_probe_record(task: dict, gold: dict, snapshot_tgz: str | Path) -> tuple[dict | None, str]:
    """Return (record, reason). record is None when the task is excluded (PLAN §9, rule 3)."""
    if len(gold.get("src_files") or []) != 1 or gold.get("new_src_files") or not gold.get("functions"):
        return None, "not eligible"
    path = gold["src_files"][0]
    texts, py = snapshot_scan(snapshot_tgz, {path})
    base = texts.get(path)
    if base is None:
        return None, "gold file missing in snapshot"
    fd = next(f for f in parse_unified_diff(task["patch"]) if f.path == path)
    scopes = {s.qualname: s for s in collect_scopes(base)}
    edit = fn = None
    for f in sorted(gold["functions"], key=lambda f: f["def_line"]):
        sc = scopes.get(f["qualname"])
        if sc is None:
            continue
        e = canonical_edit(base, fd, sc.start, sc.end)
        if e is not None:
            edit, fn = e, (f, sc)
            break
    if edit is None:
        return None, "no canonical edit"
    f, sc = fn
    total = len(base.splitlines())
    rs, re_ = sc.start, min(sc.end, sc.start + MAX_READ_LINES - 1)
    if edit["changed_lines"][1] > re_:
        return None, "change outside read span"
    names = required_test_names(task["test_patch"])
    tfiles = [fd2.path for fd2 in parse_unified_diff(task["test_patch"]) if fd2.new_path]
    rec = {
        "instance_id": task["instance_id"], "repo": task["repo"], "base_commit": task["base_commit"],
        "problem_statement": task["problem_statement"], "hints_text": task.get("hints_text") or "",
        "gold": {"file": path, "qualname": f["qualname"], "dotted": f["dotted"], "kind": f["kind"],
                 "start_line": sc.start, "def_line": sc.def_line, "end_line": sc.end, "file_total_lines": total},
        "find_target": {"tool": "read_file", "args": {"filepath": path, "start_line": rs, "end_line": re_}},
        "read_output": harness_read_output(path, base, rs, re_),
        "edit_target": {"tool": "edit_file", "args": {"filepath": path, "old_string": edit["old_string"],
                                                      "new_string": edit["new_string"]}},
        "edit_region": [edit["old_start"], edit["old_end"]], "changed_lines": edit["changed_lines"],
        "outside_edits": edit["outside_edits"],
        "diff": file_diff_text(task["patch"], path),
        "test": {"files": tfiles, "names": names, "files_exist": not (set(gold.get("new_test_files") or []) >= set(tfiles)),
                 "source": test_source(task["test_patch"])},
        "graph": None, "words": None, "wrong": {},
        "_base_text": base, "_py_files": py,  # private helpers, dropped before saving
    }
    rec["wrong"].update(_wrong_location(rec, base, py))
    return rec, "ok"


# --------------------------------------------------------------------------- rung texts

def loc_file_text(path: str) -> str:
    return f"{LOC_HEADER}\n- The fix changes `{path}`."


def loc_func_text(path: str, dotted: str, kind: str, def_line: int) -> str:
    what = "class body of" if kind == "class" else "function"
    return f"{loc_file_text(path)}\n- In `{path}`: {what} `{dotted}` (defined at line {def_line})."


def test_text(rec: dict, source: str | None = None, files: list[str] | None = None, exists: bool | None = None) -> str:
    files = rec["test"]["files"] if files is None else files
    exists = rec["test"]["files_exist"] if exists is None else exists
    status = "exists" if exists else "does not exist yet; it is added when your fix is verified"
    src = rec["test"]["source"] if source is None else source
    return (f"## Reproduction test (verified)\nThe fix is verified by this test from `{', '.join(files)}` "
            f"(the file {status}):\n```python\n{src}\n```")


def words_text(words: str) -> str:
    return f"## The change, in words\n{words}"


def diff_text(diff: str) -> str:
    return f"## The exact change\n```diff\n{diff}\n```"


def rung_text(rec: dict, cond: str) -> str | None:
    """Text appended to the problem statement for `cond` (None = graph-only or none). None if unavailable."""
    g, w = rec["gold"], rec["wrong"]
    words = rec.get("words")
    parts: list[str] = []
    if cond in ("L1", "L2", "L3", "L4", "L5"):
        parts.append(loc_file_text(g["file"]) if cond == "L1" else loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"]))
    if cond in ("L3", "L4", "L5", "T"):
        parts.append(test_text(rec))
    if cond in ("L4", "L5", "W"):
        if not words:
            return None
        parts.append(words_text(words))
    if cond == "L5":
        parts.append(diff_text(rec["diff"]))
    if cond == "X-file":
        parts.append(loc_file_text(w["file"]))
    if cond == "X-func":
        parts.append(loc_func_text(g["file"], w["dotted"], w["kind"], w["def_line"]))
    if cond == "X-test":
        if not w.get("test_source"):
            return None
        parts += [loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"]),
                  test_text(rec, w["test_source"], w["test_files"], w["test_exists"])]
    if cond == "X-words":
        if not w.get("words") or not words:
            return None
        parts += [loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"]), test_text(rec), words_text(w["words"])]
    return "\n\n".join(parts) if parts else ""


def insert_hint(user_text: str, problem_statement: str, hint: str) -> tuple[str, str]:
    """Append `hint` to the problem statement inside the harness task prompt, exactly as
    pilot_c.hints.rewrite_task would have produced it. Returns (text, method)."""
    if not hint:
        return user_text, "none"
    new_ps = problem_statement.rstrip() + "\n\n" + hint + "\n"
    if problem_statement in user_text:
        return user_text.replace(problem_statement, new_ps, 1), "exact"
    m = re.search(r"\n\n## ", user_text[user_text.find("Problem Statement:"):])
    if m:
        k = user_text.find("Problem Statement:") + m.start()
        return user_text[:k] + "\n\n" + hint + user_text[k:], "before-next-section"
    return user_text + "\n\n" + hint, "append"


# --------------------------------------------------------------------------- graph encodings

def encode_graph(focal: str, nodes: list[str], edges: list[dict], fmt: str) -> str:
    """Same content, three layouts (PLAN §4.2). nodes/edges as from get_code_subgraph."""
    nodes = [focal] + [n for n in nodes if n != focal]
    if fmt == "json":
        return json.dumps({"status": "ok", "nodes": nodes, "edges": edges, "node_count": len(nodes),
                           "edge_count": len(edges), "is_truncated": False}, ensure_ascii=False)
    if fmt == "edges":
        lines = [f"{e['from']} -[{e.get('type', 'calls')}]-> {e['to']}" for e in edges]
        return "\n".join(lines) if lines else f"{focal} (no edges)"
    if fmt == "adj":
        out = []
        for n in nodes:
            calls = [e["to"] for e in edges if e["from"] == n]
            by = [e["from"] for e in edges if e["to"] == n]
            out.append(f"{n}: calls {', '.join(calls) or '-'}; called by {', '.join(by) or '-'}")
        return "\n".join(out)
    raise ValueError(fmt)


# --------------------------------------------------------------------------- wrong controls

def _closest(target_len: int, cands: list[tuple[str, Any]], render) -> tuple[Any, float] | None:
    best = None
    for key, val in sorted(cands, key=lambda kv: kv[0]):
        r = len(render(val)) / max(1, target_len)
        d = abs(r - 1)
        if best is None or d < best[2]:
            best = (val, r, d)
    return (best[0], round(best[1], 3)) if best else None


def _wrong_location(rec: dict, base: str, py: list[str]) -> dict:
    g = rec["gold"]
    out: dict[str, Any] = {}
    files = [p for p in py if p != g["file"] and p.split("/")[0] == g["file"].split("/")[0]] or [p for p in py if p != g["file"]]
    pick = _closest(len(loc_file_text(g["file"])), [(p, p) for p in files], loc_file_text)
    if pick:
        out["file"], out["file_len_ratio"] = pick
    mod = module_path(g["file"])
    gold_q = g["qualname"]
    cands = []
    for s in collect_scopes(base):
        rs = report_scope([s])
        if s.qualname == gold_q or gold_q.startswith(s.qualname + ".") or s.qualname.startswith(gold_q + "."):
            continue
        if rs is not s or s.qualname.count(".") > 1:
            continue
        cands.append((s.qualname, {"dotted": f"{mod}.{s.qualname}" if mod else s.qualname, "kind": s.kind,
                                   "def_line": s.def_line, "qualname": s.qualname}))
    pick = _closest(len(loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"])), cands,
                    lambda v: loc_func_text(g["file"], v["dotted"], v["kind"], v["def_line"]))
    if pick:
        v, r = pick
        out.update({"dotted": v["dotted"], "kind": v["kind"], "def_line": v["def_line"], "qualname": v["qualname"],
                    "func_len_ratio": r})
    return out


def assign_cross_task_controls(records: list[dict]) -> None:
    """X-test and X-words donors: another record of the same repo with the closest text length."""
    for rec in records:
        same = [r for r in records if r["repo"] == rec["repo"] and r["instance_id"] != rec["instance_id"]]
        pick = _closest(len(rec["test"]["source"]), [(r["instance_id"], r) for r in same], lambda r: r["test"]["source"])
        if pick:
            d, ratio = pick
            rec["wrong"].update({"test_from": d["instance_id"], "test_source": d["test"]["source"],
                                 "test_files": d["test"]["files"], "test_exists": d["test"]["files_exist"],
                                 "test_len_ratio": ratio})
        if rec.get("words"):
            wsame = [(r["instance_id"], r) for r in same if r.get("words")]
            pick = _closest(len(rec["words"]), wsame, lambda r: r["words"])
            if pick:
                d, ratio = pick
                rec["wrong"].update({"words_from": d["instance_id"], "words": d["words"], "words_len_ratio": ratio})


# --------------------------------------------------------------------------- words rung

def words_prompt(diff: str, strict: bool = False) -> str:
    extra = " Do not copy any code, expression or string literal from the diff." if strict else ""
    return ("Describe the following code change in 1 to 3 plain English sentences for a developer who has to "
            "re-implement it. Say what behaviour changes and where. You may name functions or parameters, but do not "
            f"write code.{extra}\n\n```diff\n{diff}\n```")


def words_leak(words: str, diff: str, min_len: int = 20) -> bool:
    if len(words) > 600:
        return True
    plus = [l[1:].strip() for l in diff.splitlines() if l.startswith("+") and not l.startswith("+++")]
    for l in plus:
        for i in range(0, max(0, len(l) - min_len) + 1):
            if len(l) >= min_len and l[i:i + min_len] in words:
                return True
    return False


# --------------------------------------------------------------------------- depth-2 classification

def _norm_path(p: Any) -> str:
    p = str(p or "").strip().lstrip("/")
    return p[len("workspace/"):] if p.startswith("workspace/") else p


def _parse_args(a: Any) -> dict:
    if isinstance(a, dict):
        return a
    try:
        v = json.loads(a or "{}")
        return v if isinstance(v, dict) else {}
    except (ValueError, TypeError):
        return {}


def classify_find(rec: dict, tool_calls: list[dict]) -> dict:
    g, w = rec["gold"], rec["wrong"]
    out = {"tool_call": bool(tool_calls), "hit_file": False, "hit_func": False, "followed_wrong": False, "tool": None}
    if not tool_calls:
        return out
    fn = tool_calls[0].get("function", {})
    name, args = fn.get("name"), _parse_args(fn.get("arguments"))
    out["tool"] = name

    def touches(path: str, lo: int | None, hi: int | None) -> tuple[bool, bool]:
        if name == "read_file" and _norm_path(args.get("filepath")) == path:
            s = int(args.get("start_line") or 1)
            e = int(args.get("end_line") or s + MAX_READ_LINES - 1)
            return True, (lo is not None and not (e < lo or s > hi))
        if name == "run_command" and path in str(args.get("command", "")):
            return True, False
        return False, False

    out["hit_file"], out["hit_func"] = touches(g["file"], g["start_line"], g["end_line"])
    if name in ("get_code_neighbors", "get_code_subgraph", "search_similar_code"):
        blob = json.dumps(args)
        out["hit_func"] = g["qualname"].split(".")[-1] in blob
    if w.get("file"):
        out["followed_wrong"] = touches(w["file"], None, None)[0]
    if w.get("qualname"):
        out["followed_wrong"] = out["followed_wrong"] or (w["qualname"].split(".")[-1] in json.dumps(args))
    return out


def classify_use(rec: dict, tool_calls: list[dict]) -> tuple[dict, str | None]:
    """Returns (flags, edited_file_text or None). Edited text is private (for CPU test verification)."""
    base = rec["_base_text"]
    g = rec["gold"]
    out = {"tool_call": bool(tool_calls), "tool": None, "edit_gold_file": False, "applies": False,
           "on_target": False, "exact": False, "followed_wrong": False}
    if not tool_calls:
        return out, None
    fn = tool_calls[0].get("function", {})
    name, args = fn.get("name"), _parse_args(fn.get("arguments"))
    out["tool"] = name
    path = _norm_path(args.get("filepath"))
    if name not in ("edit_file", "write_file") or path != g["file"]:
        if rec["wrong"].get("file") and path == rec["wrong"]["file"]:
            out["followed_wrong"] = True
        return out, None
    out["edit_gold_file"] = True
    if name == "edit_file":
        edited, how = apply_edit(base, str(args.get("old_string", "")), str(args.get("new_string", "")))
    else:
        edited, how = str(args.get("content", "")), "write"
    if edited is None:
        return out, None
    out["applies"] = True
    gold_edited, _ = apply_edit(base, rec["edit_target"]["args"]["old_string"], rec["edit_target"]["args"]["new_string"])
    bl, el = base.splitlines(), edited.splitlines()
    i = 0
    while i < min(len(bl), len(el)) and bl[i] == el[i]:
        i += 1
    j = 0
    while j < min(len(bl), len(el)) - i and bl[-1 - j] == el[-1 - j]:
        j += 1
    lo, hi = i + 1, len(bl) - j
    ga, gb = rec["changed_lines"]
    out["on_target"] = not (hi < ga - 1 or lo > gb + 1)
    out["exact"] = gold_edited is not None and _norm(edited) == _norm(gold_edited)
    if rec["wrong"].get("def_line") and rec["wrong"].get("qualname"):
        wl = rec["wrong"]["def_line"]
        out["followed_wrong"] = lo <= wl + 5 and hi >= wl and not out["on_target"]
    return out, edited


def public_record(rec: dict) -> dict:
    """Record without private helper fields (still competition-derived: keep on Kaggle)."""
    return {k: v for k, v in rec.items() if not k.startswith("_")}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/capture_server.py
"""Capture server: an OpenAI-compatible stand-in LLM that records the agent's exact requests.

Used by the probe (docs/PLAN.md §3) to obtain byte-identical agent contexts without a GPU:
the real swegemma agent loop runs against this server; for each task the server answers with a
scripted action sequence (read the gold span, then submit_patch, then stop) and stores the full
request bodies. Request 0 is the S-find context, request 1 the S-use context (it contains the real
harness read_file output). Summarizer (no-tools) requests get a short text reply.
"""
from __future__ import annotations

import hashlib
import json
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any


def _text(m: dict) -> str:
    c = m.get("content")
    if isinstance(c, list):
        return " ".join(str(x.get("text", "")) for x in c if isinstance(x, dict))
    return str(c or "")


class CaptureServer:
    def __init__(self, routes: dict[str, dict[str, Any]], port: int = 0):
        """routes: task_id -> {"needle": str found in the task prompt, "script": [ {name, arguments} | None ]}"""
        self.routes = routes
        self.captured: dict[str, list[dict]] = {k: [] for k in routes}
        self.unrouted = 0
        self.lock = threading.Lock()
        outer = self

        class H(BaseHTTPRequestHandler):
            def log_message(self, *a):
                pass

            def _send(self, obj: dict) -> None:
                b = json.dumps(obj).encode()
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(b)))
                self.end_headers()
                self.wfile.write(b)

            def do_GET(self):
                self._send({"object": "list", "data": [{"id": "capture", "object": "model"}]})

            def do_POST(self):
                req = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
                msgs = req.get("messages", [])
                now = int(time.time())
                if not req.get("tools"):
                    return self._send(_completion("Summary.", None, now))
                first_user = next((_text(m) for m in msgs if m.get("role") == "user"), "")
                tid = next((k for k, r in outer.routes.items() if r["needle"] and r["needle"] in first_user), None)
                if tid is None:
                    with outer.lock:
                        outer.unrouted += 1
                    return self._send(_completion("Done.", None, now))
                with outer.lock:
                    step = len(outer.captured[tid])
                    outer.captured[tid].append(req)
                script = outer.routes[tid]["script"]
                item = script[step] if step < len(script) else None
                if item is None:
                    return self._send(_completion("Done.", None, now))
                return self._send(_completion(None, item, now))

        self.httpd = ThreadingHTTPServer(("127.0.0.1", port), H)
        threading.Thread(target=self.httpd.serve_forever, daemon=True).start()

    @property
    def url(self) -> str:
        return f"http://127.0.0.1:{self.httpd.server_address[1]}/v1"

    def stop(self) -> None:
        self.httpd.shutdown()
        self.httpd.server_close()


def _completion(text: str | None, call: dict | None, now: int) -> dict:
    if call is None:
        msg, fin = {"role": "assistant", "content": text}, "stop"
    else:
        msg = {"role": "assistant", "content": None, "tool_calls": [{
            "id": "call_" + uuid.uuid4().hex[:10], "type": "function",
            "function": {"name": call["name"], "arguments": json.dumps(call["arguments"])}}]}
        fin = "tool_calls"
    return {"id": "chatcmpl-capture", "object": "chat.completion", "created": now, "model": "capture",
            "choices": [{"index": 0, "message": msg, "finish_reason": fin}],
            "usage": {"prompt_tokens": 100, "completion_tokens": 10, "total_tokens": 110}}


def needle_for(problem_statement: str) -> str:
    """A short, distinctive substring of the problem statement as the harness inserts it."""
    ps = problem_statement.strip()
    return ps[:120] if len(ps) >= 20 else hashlib.sha1(ps.encode()).hexdigest()

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/probe_runner.py
"""Kaggle-side runner for the information-ladder probe (docs/PLAN.md). Runs only on Kaggle / dry runs.

Flow per session:
  1. records  : Level-B probe records for the valid, eligible tasks (CPU)
  2. graph    : harness get_code_neighbors / get_code_subgraph for gold and wrong symbols (CPU)
  3. capture  : real agent loop against the capture server -> exact S-find / S-use contexts (CPU)
  4. per size : start vLLM (starter-identical) -> self-check -> [31b: words] -> depth 1 -> depth 2
Used by notebooks/C03_probe__L4x4.ipynb and notebooks/C03v_test_verify__CPU.ipynb.
Outputs: /kaggle/working/C03_probe.json (numbers + task ids only; commit it),
         /kaggle/working/private/probe_*.jsonl (contexts, records, sampled edits; never commit).
"""
from __future__ import annotations

import concurrent.futures as cf
import copy
import difflib
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

from pilot_c import kaggle_runner as kr
from pilot_c import probe as pb
from common.server_watchdog import ServerFrozen, TRIPPED, tracked
from pilot_c.capture_server import CaptureServer, needle_for

AGENT_SAMPLING = {"temperature": 0.2, "top_p": 0.95, "max_tokens": 6144,
                  "chat_template_kwargs": {"enable_thinking": True}, "thinking_token_budget": 4096}
SCORE_KW = {"chat_template_kwargs": {"enable_thinking": False}}
MAX_CELL_TOKENS = 28000


# --------------------------------------------------------------------------- http

_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))  # local server: never via a proxy


# per-request timeouts (s): a 28k-token prefill takes ~10 s on 31b; sampling is <= 6,144 new tokens
TIMEOUT_TOKENIZE, TIMEOUT_SCORE, TIMEOUT_SAMPLE, TIMEOUT_WORDS = 60, 180, 900, 300


class ServerHTTPError(RuntimeError):
    """An HTTP error from the inference server, with the response body (vLLM puts the cause there)."""


def _post(url: str, body: dict, timeout: int = TIMEOUT_SCORE) -> dict:
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with tracked():  # counted by the watchdog; fails fast once the server is declared frozen
        try:
            with _OPENER.open(req, timeout=timeout) as r:
                return json.loads(r.read())
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", errors="replace")[:600]
            raise ServerHTTPError(f"HTTP {e.code} from /{url.split('/', 3)[-1]}: {detail}") from None


def _get(url: str) -> dict:
    with _OPENER.open(url, timeout=60) as r:
        return json.loads(r.read())


def server_root(server) -> str:
    base = getattr(server, "base_url", None) or "http://127.0.0.1:8000"
    base = str(base).rstrip("/")
    return base[:-3] if base.endswith("/v1") else base


# --------------------------------------------------------------------------- 1-2: records and graphs

def build_records(task_ids: list[str]) -> tuple[list[dict], dict[str, str]]:
    d = kr.data_dir()
    tasks = {t["instance_id"]: t for t in kr.load_tasks(d / "tasks.jsonl")}
    gold = kr.build_gold(task_ids)
    recs, excluded = [], {}
    for iid in task_ids:
        rec, why = pb.build_probe_record(tasks[iid], gold[iid], d / "snapshots" / f"{iid}.tgz")
        if rec is None:
            excluded[iid] = why
        else:
            recs.append(rec)
    return recs, excluded


def attach_graphs(recs: list[dict]) -> None:
    """Gold and wrong-symbol subgraphs from the harness's own graph tools (no sandbox needed)."""
    from adk_submission.registry import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.context import SwegemmaContext
    from swegemma.models import load_tasks as sg_load
    d = kr.data_dir()
    tasks = {t.instance_id: t for t in sg_load(d / "tasks.jsonl")}
    cfg = EvalConfig(tasks_path=d / "tasks.jsonl", snapshots_dir=d / "snapshots", results_dir=kr.PRIVATE / "graph_ctx",
                     submission_dir=d / "sample_submission", models=ModelRegistry(), sandbox="subprocess",
                     graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), display_mode="quiet")
    for rec in recs:
        t = tasks[rec["instance_id"]]
        ctx = SwegemmaContext(docker_manager=None, container_id="", problem_statement="", task=t, repo=t.repo,
                              graph_dir=cfg.graph_dir, embeddings_dir=cfg.embeddings_dir, budget=cfg.budget,
                              harness=cfg.harness)
        if hasattr(ctx, "start_agent_session"):
            ctx.start_agent_session()
        tools = ctx.create_tools()

        def sub(sym: str) -> dict | None:
            try:
                nb = tools["get_code_neighbors"](node=sym, max_neighbors=50)
                nb = json.loads(nb) if isinstance(nb, str) else nb
                neigh = [n.split(";", 1)[-1] for n in nb.get("neighbors", [])]
                neigh = [n for n in neigh if not n.startswith(("tests.", "test."))][:10]
                sg = tools["get_code_subgraph"](nodes=[sym] + neigh)
                sg = json.loads(sg) if isinstance(sg, str) else sg
                return {"focal": sym, "nodes": sg.get("nodes", [sym]), "edges": sg.get("edges", [])}
            except Exception as e:  # noqa: BLE001
                return {"focal": sym, "nodes": [sym], "edges": [], "error": f"{type(e).__name__}"}
        rec["graph"] = sub(rec["gold"]["dotted"])
        if rec["wrong"].get("dotted"):
            rec["wrong"]["graph"] = sub(rec["wrong"]["dotted"])


# --------------------------------------------------------------------------- 3: capture

def capture_contexts(recs: list[dict], budget: dict, concurrency: int = 6) -> dict[str, dict]:
    """Run the real agent loop (sample agent) against the capture server; return per-task contexts."""
    import litellm
    from adk_submission import discover_adapters
    from adk_submission.registry import ModelRegistry
    from adk_submission.resolvers.generation import install_litellm_reasoning_patch
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

    routes = {r["instance_id"]: {"needle": needle_for(r["problem_statement"]),
                                 "script": [{"name": "read_file", "arguments": r["find_target"]["args"]},
                                            {"name": "submit_patch", "arguments": {}}]} for r in recs}
    srv = CaptureServer(routes)
    try:
        litellm.drop_params = True
        install_litellm_reasoning_patch()
        model = "gemma-4-e4b-it-qat-w4a16-ct"  # alias only; system prompt and tools are size-independent
        agent_dir = kr.prepare_agent_dir(model, budget)
        reg = ModelRegistry()
        reg.register(model, LiteLlm(model="openai/capture", api_base=srv.url, api_key="EMPTY"))
        ad = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
        ids = [r["instance_id"] for r in recs]
        gold = {r["instance_id"]: {"src_files": [r["gold"]["file"]], "functions": []} for r in recs}
        kr.run_condition(agent_dir, reg, ad, ids, gold, "O0", 0, concurrency)
    finally:
        srv.stop()
    out = {}
    for r in recs:
        reqs = srv.captured.get(r["instance_id"], [])
        if len(reqs) >= 2:
            out[r["instance_id"]] = {"find": reqs[0]["messages"], "use": reqs[1]["messages"], "tools": reqs[0].get("tools")}
    return out


# --------------------------------------------------------------------------- message construction

def _content_text(m: dict) -> str:
    c = m.get("content")
    if isinstance(c, list):
        return "".join(str(x.get("text", "")) for x in c if isinstance(x, dict))
    return str(c or "")


def _set_content_text(m: dict, text: str) -> None:
    c = m.get("content")
    if isinstance(c, list):
        done = False
        for x in c:
            if isinstance(x, dict) and "text" in x and not done:
                x["text"], done = text, True
            elif isinstance(x, dict) and "text" in x:
                x["text"] = ""
    else:
        m["content"] = text


def _replace_leaf(obj: Any, old: str, new: str) -> tuple[Any, bool]:
    if isinstance(obj, str):
        return (new, True) if obj == old else (obj, False)
    if isinstance(obj, dict):
        hit = False
        out = {}
        for k, v in obj.items():
            out[k], h = _replace_leaf(v, old, new)
            hit = hit or h
        return out, hit
    if isinstance(obj, list):
        res = [_replace_leaf(v, old, new) for v in obj]
        return [x for x, _ in res], any(h for _, h in res)
    return obj, False


def rewrap_tool_content(captured: str, old_payload: str, new_payload: str) -> str | None:
    """Put `new_payload` into a tool message exactly where the harness put `old_payload`."""
    if captured == old_payload:
        return new_payload
    try:
        dec = json.loads(captured)
    except ValueError:
        return captured.replace(old_payload, new_payload) if old_payload in captured else None
    new, hit = _replace_leaf(dec, old_payload, new_payload)
    if not hit:
        try:
            new, hit = _replace_leaf(dec, json.dumps(json.loads(old_payload)), new_payload)
        except ValueError:
            pass
    if not hit:
        return None
    for ea in (False, True):
        if json.dumps(dec, ensure_ascii=ea) == captured:
            return json.dumps(new, ensure_ascii=ea)
    return json.dumps(new, ensure_ascii=False)


def context_check(rec: dict, ctx: dict) -> dict:
    """Is the captured S-use tool message the harness read of the gold span, as the record predicts?"""
    use = ctx["use"]
    tool_msg = next((m for m in reversed(use) if m.get("role") == "tool"), None)
    ok = tool_msg is not None and rewrap_tool_content(_content_text(tool_msg), rec["read_output"], "X") is not None
    return {"read_output_matches": ok, "n_find_msgs": len(ctx["find"]), "n_use_msgs": len(use)}


def build_messages(rec: dict, ctx: dict, state: str, cond: str) -> tuple[list[dict], dict, str] | None:
    """(messages without target, target assistant message, hint insertion method) or None if the cell is unavailable."""
    base = copy.deepcopy(ctx["find"] if state == "find" else ctx["use"])
    ui = next(i for i, m in enumerate(base) if m.get("role") == "user")
    method = "none"
    if cond in pb.GRAPH_CONDS:
        g = rec["graph"] if cond != "X-graph" else rec["wrong"].get("graph")
        if not g:
            return None
        payload = pb.encode_graph(g["focal"], g["nodes"], g["edges"], pb.GRAPH_CONDS[cond])
        a_tpl = next(m for m in ctx["use"] if m.get("role") == "assistant" and m.get("tool_calls"))
        t_tpl = next(m for m in ctx["use"] if m.get("role") == "tool")
        a = copy.deepcopy(a_tpl)
        a["tool_calls"] = [copy.deepcopy(a_tpl["tool_calls"][0])]
        a["tool_calls"][0]["id"] = "call_probe_graph"
        a["tool_calls"][0]["function"] = {"name": "get_code_subgraph", "arguments": json.dumps({"nodes": [g["focal"]] + [n for n in g["nodes"] if n != g["focal"]]})}
        t = copy.deepcopy(t_tpl)
        if "tool_call_id" in t:
            t["tool_call_id"] = "call_probe_graph"
        if "name" in t:
            t["name"] = "get_code_subgraph"
        wrapped = rewrap_tool_content(_content_text(t_tpl), rec["read_output"], payload)
        if wrapped is None:
            return None
        _set_content_text(t, wrapped)
        base[ui + 1:ui + 1] = [a, t]
        method = "graph-turn"
    else:
        hint = pb.rung_text(rec, cond)
        if hint is None:
            return None
        text, method = pb.insert_hint(_content_text(base[ui]), rec["problem_statement"], hint)
        _set_content_text(base[ui], text)
    tgt_spec = rec["find_target"] if state == "find" else rec["edit_target"]
    a_tpl = next(m for m in ctx["use"] if m.get("role") == "assistant" and m.get("tool_calls"))
    target = copy.deepcopy(a_tpl)
    target["tool_calls"] = [copy.deepcopy(a_tpl["tool_calls"][0])]
    target["tool_calls"][0]["id"] = "call_probe_target"
    target["tool_calls"][0]["function"] = {"name": tgt_spec["tool"], "arguments": json.dumps(tgt_spec["args"], ensure_ascii=False)}
    return base, target, method


# --------------------------------------------------------------------------- scoring and sampling

def _lcp(a: list[int], b: list[int]) -> int:
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


def score(root: str, model: str, msgs: list[dict], tools: Any, target: dict) -> dict:
    """Sum of log-probabilities of the target assistant turn, as the model sees it when generating.

    P = context + generation prompt (what the model conditions on), N = context alone, F = context + target turn.
    "direct": F starts with P (always after a tool response, where the generation prompt adds nothing; and for the
    e2b/e4b template); S = F, target T = F[len(P):].
    "spliced": the 31b and 12b templates add an empty thought block to the generation prompt after a user turn, which
    the rendered assistant turn does not contain. Then A = F[len(N):] (turn header + call), G = P[len(N):]
    (header + empty thought), T = A minus the shared header, and S = P + T: what the model conditions on, plus the call."""
    body = {"model": model, "messages": msgs, "tools": tools, **SCORE_KW}
    P = _post(f"{root}/tokenize", {**body, "add_generation_prompt": True}, TIMEOUT_TOKENIZE)["tokens"]
    N = _post(f"{root}/tokenize", {**body, "add_generation_prompt": False}, TIMEOUT_TOKENIZE)["tokens"]
    F = _post(f"{root}/tokenize", {**body, "messages": msgs + [target], "add_generation_prompt": False},
              TIMEOUT_TOKENIZE)["tokens"]
    if len(F) > len(P) and F[:len(P)] == P:  # e.g. after a tool response (P == N), or templates without an empty thought
        T, S, mode = F[len(P):], F, "direct"
    elif F[:len(N)] == N and P[:len(N)] == N:
        A, G = F[len(N):], P[len(N):]
        h = _lcp(A, G)
        if h == 0 or h >= len(A):
            return {"ok": False, "why": "prefix-mismatch", "n_ctx": len(P)}
        T = A[h:]
        S, mode = P + T, "spliced"
    else:
        return {"ok": False, "why": "prefix-mismatch", "n_ctx": len(P)}
    if len(S) > MAX_CELL_TOKENS:  # PLAN §9 rule 4
        return {"ok": False, "why": "too-long", "n_ctx": len(P)}
    r = _post(f"{root}/v1/completions", {"model": model, "prompt": S, "max_tokens": 1, "temperature": 0,
                                         "prompt_logprobs": 0}, TIMEOUT_SCORE)
    pl = r["choices"][0].get("prompt_logprobs")
    if not pl or len(pl) != len(S):
        return {"ok": False, "why": "no-prompt-logprobs", "n_ctx": len(P)}
    lps = []
    for i in range(len(P), len(S)):
        e = pl[i] or {}
        v = e.get(str(S[i])) or e.get(S[i])
        if v is None:
            return {"ok": False, "why": "token-missing", "n_ctx": len(P)}
        lps.append(float(v["logprob"] if isinstance(v, dict) else v))
    return {"ok": True, "logp": round(sum(lps), 4), "n_tok": len(lps), "mean_logp": round(sum(lps) / len(lps), 5),
            "n_ctx": len(P), "mode": mode}


def sample(root: str, model: str, msgs: list[dict], tools: Any, n: int, seed: int) -> list[dict]:
    r = _post(f"{root}/v1/chat/completions", {"model": model, "messages": msgs, "tools": tools, "n": n, "seed": seed,
                                               **AGENT_SAMPLING}, TIMEOUT_SAMPLE)
    out = []
    for ch in r.get("choices", []):
        m = ch.get("message") or {}
        out.append({"tool_calls": m.get("tool_calls") or [], "finish": ch.get("finish_reason"),
                    "completion_tokens": (r.get("usage") or {}).get("completion_tokens")})
    return out


def write_words(root: str, model: str, recs: list[dict]) -> dict[str, str]:
    status = {}
    for rec in recs:
        rec["words"] = None
        for strict in (False, True):
            r = _post(f"{root}/v1/chat/completions", {"model": model, "temperature": 0, "max_tokens": 300,
                      "messages": [{"role": "user", "content": pb.words_prompt(rec["diff"], strict)}],
                      "chat_template_kwargs": {"enable_thinking": False}}, TIMEOUT_WORDS)
            w = ((r["choices"][0].get("message") or {}).get("content") or "").strip()
            if w and not pb.words_leak(w, rec["diff"]):
                rec["words"] = w
                break
        status[rec["instance_id"]] = "ok" if rec["words"] else "leak/empty"
    return status


# --------------------------------------------------------------------------- the per-size loop

def cells_for(rec: dict) -> tuple[list[tuple], list[tuple]]:
    d1 = [("find", c, 1) for c in pb.FIND_CONDS] + [("use", c, 1) for c in pb.USE_CONDS] + [("find", "L0", 2), ("use", "L0", 2)]
    d2 = [("find", c, 1) for c in pb.D2_FIND] + [("use", c, 2) for c in pb.D2_USE]
    return d1, d2


def run_size(server, model_alias: str, recs: list[dict], ctxs: dict[str, dict], deadline: float, workers: int = 16,
             write_words_here: bool = False) -> dict[str, Any]:
    root = server_root(server)
    mid = _get(f"{root}/v1/models")["data"][0]["id"]
    size = kr.SHORT[model_alias]
    out: dict[str, Any] = {"size": size, "d1": [], "d2": [], "words_status": None, "skipped": [], "selfcheck": None}
    usable = [r for r in recs if r["instance_id"] in ctxs]
    if not usable:
        raise RuntimeError("SELFCHECK failed: no captured agent contexts (the capture phase produced nothing)")
    # self-check before spending time: score one S-use and one S-find cell, and draw one short sample
    r0 = usable[0]
    try:
        checks = {}
        for st in ("use", "find"):
            m = build_messages(r0, ctxs[r0["instance_id"]], st, "L0")
            checks[st] = score(root, mid, m[0], ctxs[r0["instance_id"]]["tools"], m[1])
        m = build_messages(r0, ctxs[r0["instance_id"]], "use", "L0")
        smp = _post(f"{root}/v1/chat/completions", {"model": mid, "messages": m[0], "tools": ctxs[r0["instance_id"]]["tools"],
                    "n": 2, "seed": 1, **{**AGENT_SAMPLING, "max_tokens": 16}}, TIMEOUT_SAMPLE)
        checks["sample_choices"] = len(smp.get("choices", []))
    except Exception as e:  # noqa: BLE001 - any failure here means the API path is broken for this size
        raise RuntimeError(f"SELFCHECK failed on {size}: {type(e).__name__}: {str(e)[:400]}") from e
    out["selfcheck"] = checks
    if not (checks["use"].get("ok") and checks["find"].get("ok") and checks["sample_choices"] == 2):
        raise RuntimeError(f"SELFCHECK failed on {size}: {checks}")
    if write_words_here:
        out["words_status"] = write_words(root, mid, usable)
        pb.assign_cross_task_controls(usable)
    t0 = time.time()

    def d1_job(rec, st, cond, rep):
        if TRIPPED.is_set() or time.time() > deadline:
            return {"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "rep": rep, "ok": False,
                    "why": "server_frozen" if TRIPPED.is_set() else "time_guard"}
        mm = build_messages(rec, ctxs[rec["instance_id"]], st, cond)
        if mm is None:
            return {"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "rep": rep, "ok": False, "why": "unavailable"}
        res = score(root, mid, mm[0], ctxs[rec["instance_id"]]["tools"], mm[1])
        return {"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "rep": rep, "hint_method": mm[2], **res}

    jobs = [(rec, st, c, rep) for rec in usable for (st, c, rep) in cells_for(rec)[0]]
    with cf.ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(d1_job, *j) for j in jobs]
        for f in cf.as_completed(futs):
            try:
                out["d1"].append(f.result())
            except Exception as e:  # noqa: BLE001
                why = "server_frozen" if isinstance(e, ServerFrozen) or TRIPPED.is_set() else f"{type(e).__name__}: {str(e)[:120]}"
                out["d1"].append({"size": size, "ok": False, "why": why})
    out["d1_minutes"] = round((time.time() - t0) / 60, 2)
    samples_private: list[dict] = []
    if TRIPPED.is_set():
        out["skipped"].append("depth2 (server frozen)")
        return out
    if time.time() > deadline:
        out["skipped"].append("depth2 (time guard)")
        return out
    t1 = time.time()

    def d2_job(rec, st, cond, n):
        if TRIPPED.is_set() or time.time() > deadline:
            return [], [{"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "ok": False,
                         "why": "server_frozen" if TRIPPED.is_set() else "time_guard"}]
        mm = build_messages(rec, ctxs[rec["instance_id"]], st, cond)
        if mm is None:
            return [], [{"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "ok": False, "why": "unavailable"}]
        rows, priv = [], []
        for k, smp in enumerate(sample(root, mid, mm[0], ctxs[rec["instance_id"]]["tools"], n, seed=1)):
            if st == "find":
                flags, edited = pb.classify_find(rec, smp["tool_calls"]), None
            else:
                flags, edited = pb.classify_use(rec, smp["tool_calls"])
            rows.append({"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "sample": k, "ok": True,
                         "finish": smp["finish"], **flags})
            if edited is not None:
                diff = "".join(difflib.unified_diff(rec["_base_text"].splitlines(True), edited.splitlines(True),
                                                    f"a/{rec['gold']['file']}", f"b/{rec['gold']['file']}"))
                priv.append({"task": rec["instance_id"], "size": size, "cond": cond, "sample": k, "patch": diff})
        return priv, rows

    jobs2 = [(rec, st, c, n) for rec in usable for (st, c, n) in cells_for(rec)[1]]
    with cf.ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(d2_job, *j) for j in jobs2]
        for f in cf.as_completed(futs):
            try:
                priv, rows = f.result()
                out["d2"] += rows
                samples_private += priv
            except Exception as e:  # noqa: BLE001
                why = "server_frozen" if isinstance(e, ServerFrozen) or TRIPPED.is_set() else f"{type(e).__name__}: {str(e)[:120]}"
                out["d2"].append({"size": size, "ok": False, "why": why})
    out["d2_minutes"] = round((time.time() - t1) / 60, 2)
    kr.PRIVATE.mkdir(parents=True, exist_ok=True)
    with open(kr.PRIVATE / f"probe_samples_{size}.jsonl", "w") as f:
        for s in samples_private:
            f.write(json.dumps(s) + "\n")
    return out


def summarize(d1: list[dict], d2: list[dict]) -> dict[str, Any]:
    """Quick in-notebook summary (medians of paired deltas); full statistics run offline from the rows."""
    import statistics
    out: dict[str, Any] = {}
    by = {}
    for r in d1:
        if r.get("ok"):
            by[(r["size"], r["state"], r["cond"], r["rep"], r["task"])] = r["logp"]
    sizes = sorted({k[0] for k in by})
    for s in sizes:
        for st in ("find", "use"):
            conds = sorted({k[2] for k in by if k[0] == s and k[1] == st})
            for c in conds:
                ref = {"X-file": "L1", "X-func": "L2", "X-graph": "G-json", "X-test": "L3", "X-words": "L4"}.get(c, "L0")
                ds = [by[(s, st, c, 1, t)] - by[(s, st, ref, 1, t)] for (s2, st2, c2, rep, t) in by
                      if s2 == s and st2 == st and c2 == c and rep == 1 and (s, st, ref, 1, t) in by]
                if ds and c != ref:
                    out[f"{s}/{st}/{c}-vs-{ref}"] = {"n": len(ds), "median_delta_nats": round(statistics.median(ds), 3)}
            noise = [abs(by[(s, st, "L0", 2, t)] - by[(s, st, "L0", 1, t)]) for (s2, st2, c2, rep, t) in by
                     if s2 == s and st2 == st and c2 == "L0" and rep == 2 and (s, st, "L0", 1, t) in by]
            if noise:
                out[f"{s}/{st}/L0-rerun-abs-diff"] = {"n": len(noise), "max": round(max(noise), 4)}
    for s in sorted({r["size"] for r in d2 if r.get("ok")}):
        for st, keys in (("find", ("hit_file", "hit_func", "followed_wrong")), ("use", ("on_target", "exact", "applies", "followed_wrong"))):
            for c in sorted({r["cond"] for r in d2 if r.get("ok") and r["size"] == s and r["state"] == st}):
                rows = [r for r in d2 if r.get("ok") and r["size"] == s and r["state"] == st and r["cond"] == c]
                out[f"{s}/{st}/{c}/depth2"] = {"n": len(rows), "tool_call": round(sum(r["tool_call"] for r in rows) / len(rows), 3),
                                              **{k: round(sum(bool(r.get(k)) for r in rows) / len(rows), 3) for k in keys}}
    return out


# --------------------------------------------------------------------------- CPU verification (notebooks/C03v_test_verify__CPU.ipynb)

def verify_samples(records_path: Path, sample_files: list[Path], out_path: Path, timeout_s: int = 300) -> dict[str, Any]:
    """Apply each sampled S-use edit alone to the base snapshot and run the scorer's Phase-2 verification.
    Writes numbers only: (task, size, cond, sample, solves)."""
    from adk_submission.registry import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.deduplication import resolve_task_snapshot_paths
    from swegemma.evaluate import Evaluator
    from swegemma.harness.verification import verify_task
    from swegemma.models import load_tasks as sg_load

    d = kr.data_dir()
    cfg = EvalConfig(tasks_path=d / "tasks.jsonl", snapshots_dir=d / "snapshots", results_dir=kr.PRIVATE / "probe_verify",
                     submission_dir=d / "sample_submission", models=ModelRegistry(), sandbox="subprocess",
                     timeout_seconds=timeout_s, verification_timeout_seconds=timeout_s, wheels_dir=d / "wheels",
                     graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), display_mode="quiet")
    ev = Evaluator(cfg)
    tasks = {t.instance_id: t for t in sg_load(d / "tasks.jsonl")}
    rows: list[dict] = []
    cache: dict[tuple[str, str], bool] = {}
    for sf in sample_files:
        for line in open(sf):
            s = json.loads(line)
            t = tasks[s["task"]]
            key = (s["task"], s["patch"])
            if key not in cache:
                snap, base, patch = resolve_task_snapshot_paths(d / "snapshots", t.instance_id, t.repo)
                try:
                    r = kr.run_sync(verify_task, ev.docker, cfg, t, snap, base_snapshot_path=base, patch_path=patch,
                                    agent_patch=s["patch"], agent_error=None, trace=None, start_time=time.perf_counter())
                    cache[key] = bool(r.resolved) and bool(s["patch"].strip())
                except Exception:  # noqa: BLE001
                    cache[key] = False
            rows.append({"task": s["task"], "size": s["size"], "cond": s["cond"], "sample": s["sample"], "solves": cache[key]})
            out_path.write_text(json.dumps({"rows": rows}, indent=1))
    summ: dict[str, Any] = {}
    for r in rows:
        k = f"{r['size']}/{r['cond']}"
        a = summ.setdefault(k, {"n": 0, "solves": 0})
        a["n"] += 1
        a["solves"] += int(r["solves"])
    res = {"rows": rows, "summary": summ, "unique_patches_verified": len(cache)}
    out_path.write_text(json.dumps(res, indent=1))
    return res

In [ ]:
import os, sys, time, json
from pathlib import Path
sys.path.insert(0, '/kaggle/working/pilot')
SESSION_T0 = time.time()
from pilot_c import kaggle_runner as kr
kr.set_env()
kr.install_wheelhouse(gpu=False, only=kr.CPU_WHEELS)
kr.quiet_logs()
IDS = json.loads(Path('/kaggle/working/pilot/pilot_c/task_ids.json').read_text())
print(kr.env_info())

In [ ]:
from common.tasks import load_tasks
from pilot_c.select_tasks import eligible
all_ids = [t['instance_id'] for t in load_tasks(kr.data_dir() / 'tasks.jsonl')]
if os.environ.get('PILOT_TEST_IDS'):           # local dry runs only
    all_ids = os.environ['PILOT_TEST_IDS'].split(',')
gold_all = kr.build_gold(all_ids)              # derived data: stays in memory / private
(kr.PRIVATE).mkdir(parents=True, exist_ok=True)
(kr.PRIVATE / 'gold_all.json').write_text(json.dumps(gold_all))
first = [i for i in all_ids if eligible(gold_all[i])]
order = first + [i for i in all_ids if i not in first]
print(len(first), 'eligible tasks first, then the rest')
OUT = Path('/kaggle/working/C00_gold_check.json')
res = kr.gold_check(order, OUT)
print('gold resolves:', sum(r['gold_resolved'] for r in res['rows']), '/', len(res['rows']),
      '| valid (gold passes AND no-patch fails):', sum(bool(r.get('valid')) for r in res['rows']))
print(json.dumps(res['summary'], indent=1))
# pre-registered selection for C01 (smoke3) and C03 (task order), from THIS environment's gold check
final = json.loads(OUT.read_text())
final['selection'] = kr.select_from_gold_check(gold_all, res['rows'])
OUT.write_text(json.dumps(final, indent=1))
print('smoke3 for notebooks/C01_smoke_timing__L4x4.ipynb:', final['selection']['smoke3'])

**When it finishes:** download `/kaggle/working/C00_gold_check.json` (numbers only: per-task flags/counts, no patches or traces)
and commit it to `results/` on branch `pilot/c-bottleneck`. Everything under `/kaggle/working/private/` (hinted task copies,
patches, traces) is competition-derived: keep it on Kaggle, never commit it.